# Pythia-1B с нуля на PyTorch

Этот ноутбук реализует архитектуру Pythia-1B / GPT-NeoX вручную на PyTorch.
`transformers` используется только для токенизатора, а не для самой модели.
Веса загружаются из официального репозитория `EleutherAI/pythia-1b`.

Что проверяется:
- структура модели и число параметров;
- загрузка весов без `AutoModelForCausalLM`;
- causal forward pass и KV-cache;
- perplexity на Tiny Shakespeare из `main.ipynb`;
- скорость prefill и autoregressive decoding.

Модель Pythia-1B имеет контекст 2048 токенов, 16 слоёв, hidden size 2048 и 8 attention heads.

In [1]:
# Если зависимости ещё не установлены, выполните в отдельной ячейке или в терминале:
# %pip install -U torch transformers huggingface_hub safetensors requests tqdm

import json
import math
import re
import time
from dataclasses import dataclass
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported():
    DTYPE = torch.bfloat16
elif DEVICE.type == 'cuda':
    DTYPE = torch.float16
else:
    DTYPE = torch.float32

print('device:', DEVICE)
print('dtype:', DTYPE)
print('torch:', torch.__version__)

device: cuda
dtype: torch.bfloat16
torch: 2.14.0+cu126


In [2]:
import gc

def clear_gpu_cache():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

## 1. Конфигурация Pythia-1B

Конфигурация соответствует `config.json` официальной модели. Pythia использует GPT-NeoX block с parallel residual: attention и MLP получают нормализованный вход параллельно и затем складываются с residual.

In [3]:
@dataclass
class PythiaConfig:
    vocab_size: int = 50304
    hidden_size: int = 2048
    intermediate_size: int = 8192
    num_hidden_layers: int = 16
    num_attention_heads: int = 8
    max_position_embeddings: int = 2048
    rotary_pct: float = 0.25
    rotary_emb_base: float = 10000.0
    layer_norm_eps: float = 1e-5
    hidden_act: str = 'gelu'
    use_parallel_residual: bool = True
    use_cache: bool = True
    attention_bias: bool = True

    @property
    def head_dim(self):
        assert self.hidden_size % self.num_attention_heads == 0
        return self.hidden_size // self.num_attention_heads

    @property
    def rotary_ndims(self):
        return int(self.head_dim * self.rotary_pct)

config = PythiaConfig()
print(config)
print('head_dim:', config.head_dim)
print('rotary_ndims:', config.rotary_ndims)

PythiaConfig(vocab_size=50304, hidden_size=2048, intermediate_size=8192, num_hidden_layers=16, num_attention_heads=8, max_position_embeddings=2048, rotary_pct=0.25, rotary_emb_base=10000.0, layer_norm_eps=1e-05, hidden_act='gelu', use_parallel_residual=True, use_cache=True, attention_bias=True)
head_dim: 256
rotary_ndims: 64


## 2. Rotary position embeddings

В Pythia/GPT-NeoX rotary embedding применяется только к первым 25% размерности attention head. Остальные координаты query и key проходят без rotary-преобразования.

In [4]:
def rotate_half(x):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)


class RotaryEmbedding(nn.Module):
    def __init__(self, dim, base=10000.0):
        super().__init__()
        self.dim = dim
        self.base = base
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq, persistent=False)

    def forward(self, position_ids, dtype):
        # position_ids: [sequence_length]
        freqs = torch.outer(position_ids.float(), self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        cos = emb.cos().to(dtype=dtype)[None, None, :, :]
        sin = emb.sin().to(dtype=dtype)[None, None, :, :]
        return cos, sin


def apply_rotary(q, k, cos, sin, rotary_ndims):
    q_rot, q_pass = q[..., :rotary_ndims], q[..., rotary_ndims:]
    k_rot, k_pass = k[..., :rotary_ndims], k[..., rotary_ndims:]
    q_rot = q_rot * cos + rotate_half(q_rot) * sin
    k_rot = k_rot * cos + rotate_half(k_rot) * sin
    return torch.cat((q_rot, q_pass), dim=-1), torch.cat((k_rot, k_pass), dim=-1)

## 3. Attention, MLP и Transformer block

Здесь сохранены имена модулей GPT-NeoX. Благодаря этому state dict из Hugging Face можно загрузить напрямую, без ручного переименования каждого слоя.

In [5]:
class PythiaAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.num_attention_heads = config.num_attention_heads
        self.head_dim = config.head_dim
        self.rotary_ndims = config.rotary_ndims
        self.query_key_value = nn.Linear(
            config.hidden_size,
            3 * config.hidden_size,
            bias=config.attention_bias,
        )
        self.dense = nn.Linear(
            config.hidden_size,
            config.hidden_size,
            bias=config.attention_bias,
        )
        self.rotary_emb = RotaryEmbedding(config.rotary_ndims, config.rotary_emb_base)
        self.scale = self.head_dim ** -0.5

    def _causal_mask(self, query_length, key_length, past_length, device):
        # Строка i соответствует query с абсолютной позицией past_length + i.
        # Разрешены только ключи с позицией не больше позиции query.
        return torch.ones(query_length, key_length, device=device, dtype=torch.bool).tril(
            diagonal=past_length
        )

    def forward(self, hidden_states, past_key_value=None, use_cache=False):
        batch_size, query_length, _ = hidden_states.shape
        qkv = self.query_key_value(hidden_states)
        # GPT-NeoX layout: [batch, seq, heads, 3 * head_dim].
        # Нельзя использовать [batch, seq, 3, heads, head_dim]:
        # это перемешает Q/K/V между attention heads.
        qkv = qkv.view(
            batch_size, query_length, self.num_attention_heads, 3 * self.head_dim
        ).transpose(1, 2)
        query, key, value = qkv.chunk(3, dim=-1)

        past_length = 0 if past_key_value is None else past_key_value[0].shape[2]
        position_ids = torch.arange(
            past_length, past_length + query_length, device=hidden_states.device
        )
        cos, sin = self.rotary_emb(position_ids, hidden_states.dtype)
        query, key = apply_rotary(query, key, cos, sin, self.rotary_ndims)

        if past_key_value is not None:
            key = torch.cat((past_key_value[0], key), dim=2)
            value = torch.cat((past_key_value[1], value), dim=2)

        key_length = key.shape[2]
        causal_mask = self._causal_mask(
            query_length, key_length, past_length, hidden_states.device
        )
        attention_output = F.scaled_dot_product_attention(
            query, key, value,
            attn_mask=causal_mask,
            dropout_p=0.0,
            is_causal=False,
        )
        attention_output = attention_output.transpose(1, 2).contiguous()
        attention_output = attention_output.view(batch_size, query_length, -1)
        attention_output = self.dense(attention_output)

        present = (key, value) if use_cache else None
        return attention_output, present


class PythiaMLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.dense_h_to_4h = nn.Linear(
            config.hidden_size, config.intermediate_size, bias=True
        )
        self.dense_4h_to_h = nn.Linear(
            config.intermediate_size, config.hidden_size, bias=True
        )

    def forward(self, hidden_states):
        hidden_states = self.dense_h_to_4h(hidden_states)
        hidden_states = F.gelu(hidden_states)
        return self.dense_4h_to_h(hidden_states)


class PythiaDecoderLayer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.input_layernorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.post_attention_layernorm = nn.LayerNorm(
            config.hidden_size, eps=config.layer_norm_eps
        )
        self.attention = PythiaAttention(config)
        self.mlp = PythiaMLP(config)
        self.use_parallel_residual = config.use_parallel_residual

    def forward(self, hidden_states, past_key_value=None, use_cache=False):
        residual = hidden_states
        attention_input = self.input_layernorm(hidden_states)
        attention_output, present = self.attention(
            attention_input, past_key_value=past_key_value, use_cache=use_cache
        )

        if self.use_parallel_residual:
            mlp_input = self.post_attention_layernorm(hidden_states)
            mlp_output = self.mlp(mlp_input)
            hidden_states = residual + attention_output + mlp_output
        else:
            hidden_states = residual + attention_output
            hidden_states = hidden_states + self.mlp(
                self.post_attention_layernorm(hidden_states)
            )
        return hidden_states, present


class PythiaForCausalLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.gpt_neox = nn.Module()
        self.gpt_neox.embed_in = nn.Embedding(config.vocab_size, config.hidden_size)
        self.gpt_neox.layers = nn.ModuleList(
            [PythiaDecoderLayer(config) for _ in range(config.num_hidden_layers)]
        )
        self.gpt_neox.final_layer_norm = nn.LayerNorm(
            config.hidden_size, eps=config.layer_norm_eps
        )
        self.embed_out = nn.Linear(config.hidden_size, config.vocab_size, bias=False)

    def forward(self, input_ids, past_key_values=None, use_cache=False):
        hidden_states = self.gpt_neox.embed_in(input_ids)
        if past_key_values is None:
            past_key_values = [None] * len(self.gpt_neox.layers)

        presents = []
        for layer, past in zip(self.gpt_neox.layers, past_key_values):
            hidden_states, present = layer(
                hidden_states, past_key_value=past, use_cache=use_cache
            )
            if use_cache:
                presents.append(present)

        hidden_states = self.gpt_neox.final_layer_norm(hidden_states)
        logits = self.embed_out(hidden_states)
        return logits, tuple(presents) if use_cache else None

    @torch.no_grad()
    def generate_greedy(self, input_ids, max_new_tokens=64):
        self.eval()
        logits, past = self(input_ids, use_cache=True)
        generated = [input_ids]
        next_token = logits[:, -1:, :].argmax(dim=-1)
        generated.append(next_token)
        for _ in range(max_new_tokens - 1):
            logits, past = self(next_token, past_key_values=past, use_cache=True)
            next_token = logits[:, -1:, :].argmax(dim=-1)
            generated.append(next_token)
        return torch.cat(generated, dim=1)

## 4. Загрузка токенизатора и официальных весов

Мы не вызываем `AutoModelForCausalLM`. `huggingface_hub` скачивает файлы весов, `safetensors` читает их, а затем они загружаются в написанную выше модель.

In [6]:
from huggingface_hub import snapshot_download
from safetensors.torch import load_file as load_safetensors
from transformers import AutoTokenizer

MODEL_ID = 'EleutherAI/pythia-1b'
MODEL_DIR = Path(
    snapshot_download(
        repo_id=MODEL_ID,
        allow_patterns=[
            'config.json',
            'tokenizer*',
            '*.json',
            '*.safetensors',
            '*.bin',
        ],
    )
)

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), use_fast=True)
print('model directory:', MODEL_DIR)
print('tokenizer vocab:', tokenizer.vocab_size)
print('bos:', tokenizer.bos_token_id, 'eos:', tokenizer.eos_token_id)

def load_weight_file(path):
    if path.suffix == '.safetensors':
        return load_safetensors(str(path), device='cpu')
    try:
        return torch.load(path, map_location='cpu', weights_only=True)
    except TypeError:
        return torch.load(path, map_location='cpu')


def load_official_weights(model, model_dir):
    model_dir = Path(model_dir)
    index_candidates = [
        model_dir / 'model.safetensors.index.json',
        model_dir / 'pytorch_model.bin.index.json',
    ]
    state_dict = {}
    index_path = next((p for p in index_candidates if p.exists()), None)

    if index_path is not None:
        index = json.loads(index_path.read_text())
        shard_names = sorted(set(index['weight_map'].values()))
        for shard_name in shard_names:
            shard = load_weight_file(model_dir / shard_name)
            state_dict.update(shard)
            del shard
    else:
        single_candidates = [
            model_dir / 'model.safetensors',
            model_dir / 'pytorch_model.bin',
        ]
        weight_path = next((p for p in single_candidates if p.exists()), None)
        if weight_path is None:
            raise FileNotFoundError('Не найден файл весов в ' + str(model_dir))
        state_dict = load_weight_file(weight_path)

    # Некоторые версии GPT-NeoX сохраняют вспомогательные rotary buffers,
    # которых нет среди обучаемых параметров нашей реализации.
    expected_keys = set(model.state_dict().keys())
    ignored_keys = sorted(set(state_dict.keys()) - expected_keys)
    filtered_state_dict = {
        key: value for key, value in state_dict.items() if key in expected_keys
    }
    missing, unexpected = model.load_state_dict(filtered_state_dict, strict=False)
    del state_dict, filtered_state_dict
    if ignored_keys:
        print('ignored auxiliary checkpoint keys:', ignored_keys[:20])
    if missing:
        print('missing trainable keys:', missing[:20])
        print('checkpoint keys:', len(expected_keys))
        raise RuntimeError(
            'Не загружены обязательные параметры; проверьте списки missing keys'
        )
    if unexpected:
        print('unexpected keys after filtering:', unexpected[:20])
        raise RuntimeError('Неожиданные параметры в state dict')
    return model

model = PythiaForCausalLM(config)
model = load_official_weights(model, MODEL_DIR)
model = model.to(device=DEVICE, dtype=DTYPE)
model.eval()

parameter_count = sum(p.numel() for p in model.parameters())
print(f'parameters: {parameter_count:,} ({parameter_count / 1e9:.3f}B)')
print('model loaded successfully')

/home/froschin/work/llm/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 1386.16it/s]


model directory: /home/froschin/.cache/huggingface/hub/models--EleutherAI--pythia-1b/snapshots/f73d7dcc545c8bd326d8559c8ef84ffe92fea6b2
tokenizer vocab: 50254
bos: 0 eos: 0
ignored auxiliary checkpoint keys: ['gpt_neox.layers.0.attention.bias', 'gpt_neox.layers.0.attention.masked_bias', 'gpt_neox.layers.0.attention.rotary_emb.inv_freq', 'gpt_neox.layers.1.attention.bias', 'gpt_neox.layers.1.attention.masked_bias', 'gpt_neox.layers.1.attention.rotary_emb.inv_freq', 'gpt_neox.layers.10.attention.bias', 'gpt_neox.layers.10.attention.masked_bias', 'gpt_neox.layers.10.attention.rotary_emb.inv_freq', 'gpt_neox.layers.11.attention.bias', 'gpt_neox.layers.11.attention.masked_bias', 'gpt_neox.layers.11.attention.rotary_emb.inv_freq', 'gpt_neox.layers.12.attention.bias', 'gpt_neox.layers.12.attention.masked_bias', 'gpt_neox.layers.12.attention.rotary_emb.inv_freq', 'gpt_neox.layers.13.attention.bias', 'gpt_neox.layers.13.attention.masked_bias', 'gpt_neox.layers.13.attention.rotary_emb.inv_freq',

## 5. Проверка forward pass и KV-cache

Сначала прогоняем короткую последовательность. Затем проверяем, что logits последнего токена при cached decoding совпадают с logits dense-prefill в пределах погрешности выбранного dtype.

In [7]:
def max_abs_diff(a, b):
    return (a.float() - b.float()).abs().max().item()

with torch.inference_mode():
    test_ids = torch.randint(0, config.vocab_size, (1, 32), device=DEVICE)
    dense_logits, _ = model(test_ids, use_cache=False)
    prefill_logits, past = model(test_ids[:, :-1], use_cache=True)
    cached_logits, _ = model(
        test_ids[:, -1:], past_key_values=past, use_cache=True
    )

print('logits shape:', tuple(dense_logits.shape))
print('KV layers:', len(past))
print('KV shape in layer 0:', tuple(past[0][0].shape))
print('cached-vs-dense max abs diff:', max_abs_diff(
    dense_logits[:, -1:], cached_logits
))

logits shape: (1, 32, 50304)
KV layers: 16
KV shape in layer 0: (1, 8, 31, 256)
cached-vs-dense max abs diff: 0.09375


## 6. Tiny Shakespeare из `main.ipynb`

В `main.ipynb` используется датасет `tinyshakespeare/input.txt`. Следующая ячейка читает URL и имя файла из исходного ноутбука, поэтому benchmark остаётся привязанным к тому же источнику данных.

In [8]:
import requests

MAIN_NOTEBOOK = Path('/home/froschin/work/llm/main.ipynb')
WORK_DIR = MAIN_NOTEBOOK.parent

main_notebook = json.loads(MAIN_NOTEBOOK.read_text(encoding='utf-8'))
main_source = '\n'.join(
    ''.join(cell.get('source', []))
    for cell in main_notebook.get('cells', [])
)
# Эти значения совпадают с ячейкой загрузки датасета в main.ipynb.
SHAKESPEARE_URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
SHAKESPEARE_FILE = WORK_DIR / 'tinyshakespeare.txt'

if not SHAKESPEARE_FILE.exists():
    print('Downloading:', SHAKESPEARE_URL)
    response = requests.get(SHAKESPEARE_URL, timeout=60)
    response.raise_for_status()
    SHAKESPEARE_FILE.write_text(response.text, encoding='utf-8')

shakespeare_text = SHAKESPEARE_FILE.read_text(encoding='utf-8')
shakespeare_ids = tokenizer(
    shakespeare_text, add_special_tokens=False, return_tensors='pt'
).input_ids[0]

print('file:', SHAKESPEARE_FILE)
print('characters:', len(shakespeare_text))
print('tokens:', len(shakespeare_ids))
print('preview:', repr(shakespeare_text[:300]))

file: /home/froschin/work/llm/tinyshakespeare.txt
characters: 1115394
tokens: 340240
preview: "First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\nFirst Citizen:\nFirst, you know Caius Marcius is chief enemy to the people.\n\nAll:\nWe know't, we know't.\n\nFirst Citizen:\nLet us"


## 7. Benchmark: perplexity

Это benchmark pretrained-модели без дополнительного обучения на Shakespeare. Для честного сравнения Ocean нужно будет использовать те же token ids, те же окна и те же logits.

In [9]:
@torch.inference_mode()
def evaluate_perplexity(model, token_ids, max_tokens=8192, block_size=2048):
    token_ids = token_ids[:max_tokens].to(DEVICE)
    total_nll = 0.0
    total_tokens = 0
    model.eval()

    for start in range(0, len(token_ids) - 1, block_size):
        chunk = token_ids[start : start + block_size + 1]
        if len(chunk) < 2:
            continue
        input_ids = chunk[:-1].unsqueeze(0)
        targets = chunk[1:].unsqueeze(0)
        logits, _ = model(input_ids, use_cache=False)
        loss = F.cross_entropy(
            logits.float().reshape(-1, config.vocab_size),
            targets.reshape(-1),
            reduction='sum',
        )
        total_nll += loss.item()
        total_tokens += targets.numel()

    mean_nll = total_nll / total_tokens
    return mean_nll, math.exp(mean_nll)

ppl_start = time.perf_counter()
mean_nll, perplexity = evaluate_perplexity(model, shakespeare_ids)
if DEVICE.type == 'cuda':
    torch.cuda.synchronize()
ppl_elapsed = time.perf_counter() - ppl_start

print(f'mean NLL: {mean_nll:.4f}')
print(f'perplexity: {perplexity:.2f}')
print(f'evaluation time: {ppl_elapsed:.2f}s')

mean NLL: 2.9874
perplexity: 19.83
evaluation time: 1.77s


## 8. Benchmark: prefill и autoregressive decoding

`prefill` обрабатывает prompt целиком, а затем `generate_greedy` использует KV-cache и подаёт в модель по одному новому токену. Для Ocean именно второй участок удобно заменять routed attention.

In [10]:
def synchronize():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()

@torch.inference_mode()
def benchmark_generation(
    model, token_ids, prompt_length=512, new_tokens=64,
    allow_untrained_context=False,
):
    if prompt_length > config.max_position_embeddings:
        message = (
            f'prompt_length={prompt_length} превышает штатный контекст '
            f'{config.max_position_embeddings} токенов. Это экстраполяция RoPE, '
            'не проверенное расширение контекста.'
        )
        if not allow_untrained_context:
            raise ValueError(message + ' Передайте allow_untrained_context=True явно.')
        print('WARNING:', message)
    prompt = token_ids[:prompt_length].unsqueeze(0).to(DEVICE)
    model.eval()

    # Prefill измеряется отдельно: один dense forward по prompt.
    synchronize()
    prefill_start = time.perf_counter()
    logits, past = model(prompt, use_cache=True)
    synchronize()
    prefill_elapsed = time.perf_counter() - prefill_start

    next_token = logits[:, -1:, :].argmax(dim=-1)
    generated = [prompt, next_token]

    # Здесь уже есть KV-cache; измеряем только последующие one-token forwards.
    synchronize()
    decode_start = time.perf_counter()
    for _ in range(max(0, new_tokens - 1)):
        logits, past = model(next_token, past_key_values=past, use_cache=True)
        next_token = logits[:, -1:, :].argmax(dim=-1)
        generated.append(next_token)
    synchronize()
    decode_elapsed = time.perf_counter() - decode_start

    output = torch.cat(generated, dim=1)
    total_elapsed = prefill_elapsed + decode_elapsed
    decode_steps = max(1, new_tokens - 1)
    generated_text = tokenizer.decode(output[0].tolist())
    return {
        'output': generated_text,
        'prefill_seconds': prefill_elapsed,
        'prefill_tokens_per_second': prompt_length / prefill_elapsed,
        'decode_seconds': decode_elapsed,
        'decode_tokens_per_second': decode_steps / decode_elapsed,
        'total_seconds': total_elapsed,
        'total_tokens_per_second': new_tokens / total_elapsed,
    }

In [11]:
# benchmark = benchmark_generation(
#     model, shakespeare_ids, prompt_length=512, new_tokens=64
# )
# print(f'prefill seconds: {benchmark["prefill_seconds"]:.3f}')
# print(f'prefill tok/s: {benchmark["prefill_tokens_per_second"]:.2f}')
# print(f'decode seconds: {benchmark["decode_seconds"]:.3f}')
# print(f'decode tok/s: {benchmark["decode_tokens_per_second"]:.2f}')
# print(f'total seconds: {benchmark["total_seconds"]:.3f}')

# clear_gpu_cache()

prefill seconds: 0.107

prefill tok/s: 4790.68

decode seconds: 0.793

decode tok/s: 79.44

total seconds: 0.900

In [12]:
# benchmark = benchmark_generation(
#     model, shakespeare_ids, prompt_length=14_000, new_tokens=64,
#     allow_untrained_context=True,  # экспериментальная RoPE-экстраполяция
# )
# print(f'prefill seconds: {benchmark["prefill_seconds"]:.3f}')
# print(f'prefill tok/s: {benchmark["prefill_tokens_per_second"]:.2f}')
# print(f'decode seconds: {benchmark["decode_seconds"]:.3f}')
# print(f'decode tok/s: {benchmark["decode_tokens_per_second"]:.2f}')
# print(f'total seconds: {benchmark["total_seconds"]:.3f}')

# clear_gpu_cache()

prefill seconds: 6.845

prefill tok/s: 2045.15

decode seconds: 1.678

decode tok/s: 37.54

total seconds: 8.523

## 9. Ocean: routed attention для Pythia

Ниже создаётся вторая модель с теми же весами, но с заменённым attention-механизмом. В первой версии оптимизация включается только при autoregressive decoding после prefill. Это позволяет не нарушать causal semantics prefill.

Маршрут для каждого слоя и головы состоит из:
- 2 последних локальных блоков;
- 5 semantic-блоков с максимальной cosine similarity к среднему последних ключей;
- 1 deterministic exploration-блока.

Полный KV-cache сохраняется, но attention вычисляется только по выбранным блокам.

In [11]:
class OceanPythiaAttention(PythiaAttention):
    def __init__(
        self, config, layer_idx, block_size=64, summary_window=100,
        route_refresh_interval=50, local_blocks=2, semantic_blocks=5,
    ):
        super().__init__(config)
        self.layer_idx = layer_idx
        self.block_size = block_size
        self.summary_window = summary_window
        self.route_refresh_interval = route_refresh_interval
        self.local_blocks = local_blocks
        self.semantic_blocks = semantic_blocks
        self._route_cache = None
        self._route_num_blocks = None
        self._route_age = 0
        self._route_refresh_count = 0

    def reset_route(self):
        self._route_cache = None
        self._route_num_blocks = None
        self._route_age = 0
        self._route_refresh_count = 0

    def _dense_attention(self, query, key, value, past_length):
        query_length = query.shape[2]
        key_length = key.shape[2]
        mask = self._causal_mask(
            query_length, key_length, past_length, query.device
        )
        output = F.scaled_dot_product_attention(
            query, key, value, attn_mask=mask, dropout_p=0.0, is_causal=False
        )
        return output

    def _build_route(self, key):
        # Prototype рассчитан на batch=1; route различается по attention heads.
        batch_size, num_heads, key_length, _ = key.shape
        if batch_size != 1:
            raise NotImplementedError('Ocean prototype пока поддерживает только batch=1')

        num_blocks = math.ceil(key_length / self.block_size)
        padded_length = num_blocks * self.block_size
        pad = padded_length - key_length
        if pad:
            padded_key = F.pad(key, (0, 0, 0, pad))
        else:
            padded_key = key
        blocks = padded_key.view(1, num_heads, num_blocks, self.block_size, self.head_dim)
        counts = torch.full(
            (num_blocks,), self.block_size, device=key.device, dtype=key.dtype
        )
        if pad:
            counts[-1] = self.block_size - pad
        summaries = blocks.sum(dim=3) / counts.view(1, 1, num_blocks, 1)

        recent_start = max(0, key_length - self.summary_window)
        recent = key[:, :, recent_start:key_length, :].mean(dim=2)
        # recent: [1, heads, D], summaries: [1, heads, blocks, D].
        # Явное суммирование по D сохраняет отдельный score для каждого head/block.
        recent_norm = F.normalize(recent.float(), dim=-1).unsqueeze(2)
        summary_norm = F.normalize(summaries.float(), dim=-1)
        scores = (recent_norm * summary_norm).sum(dim=-1)

        local_start = max(0, num_blocks - self.local_blocks)
        local_ids = list(range(local_start, num_blocks))
        semantic_count = min(self.semantic_blocks, max(0, num_blocks - len(local_ids)))
        routes = []
        for head in range(num_heads):
            # stable=True сохраняет меньший block id при одинаковых score.
            order = torch.argsort(
                scores[0, head], descending=True, stable=True
            ).tolist()
            semantic_ids = [
                block_id for block_id in order if block_id not in local_ids
            ][:semantic_count]
            selected = local_ids + semantic_ids

            # Детерминированная псевдослучайная exploration без дубликатов.
            seed = (
                (self.layer_idx + 1) * 1000003
                + (self._route_refresh_count + 1) * 9176
                + head * 101
            )
            start = seed % num_blocks
            for offset in range(num_blocks):
                candidate = (start + offset) % num_blocks
                if candidate not in selected:
                    selected.append(candidate)
                    break
            routes.append(selected)

        route = torch.tensor(routes, device=key.device, dtype=torch.long)
        self._route_cache = route.unsqueeze(0)
        self._route_num_blocks = num_blocks
        self._route_age = 0
        self._route_refresh_count += 1
        return self._route_cache

    def _routed_attention(self, query, key, value):
        key_length = key.shape[2]
        num_blocks = math.ceil(key_length / self.block_size)
        if (
            self._route_cache is None
            or self._route_age >= self.route_refresh_interval
            or self._route_num_blocks != num_blocks
        ):
            route = self._build_route(key)
        else:
            route = self._route_cache
            self._route_age += 1

        # route: [batch=1, heads, route_blocks]. Собираем токены блоков.
        block_offsets = torch.arange(self.block_size, device=key.device)
        token_ids = route[:, :, :, None] * self.block_size + block_offsets
        valid = token_ids < key_length
        token_ids = token_ids.clamp(max=key_length - 1)
        route_tokens = token_ids.shape[2] * token_ids.shape[3]
        token_ids = token_ids.reshape(1, self.num_attention_heads, route_tokens)
        valid = valid.reshape(1, self.num_attention_heads, route_tokens)

        gather_ids = token_ids.unsqueeze(-1).expand(
            -1, -1, -1, self.head_dim
        )
        key_route = key.gather(2, gather_ids)
        value_route = value.gather(2, gather_ids)
        route_mask = valid[:, :, None, :]
        return F.scaled_dot_product_attention(
            query, key_route, value_route,
            attn_mask=route_mask, dropout_p=0.0, is_causal=False
        )

    def forward(self, hidden_states, past_key_value=None, use_cache=False):
        batch_size, query_length, _ = hidden_states.shape
        qkv = self.query_key_value(hidden_states)
        qkv = qkv.view(
            batch_size, query_length, self.num_attention_heads, 3 * self.head_dim
        ).transpose(1, 2)
        query, key, value = qkv.chunk(3, dim=-1)

        past_length = 0 if past_key_value is None else past_key_value[0].shape[2]
        position_ids = torch.arange(
            past_length, past_length + query_length, device=hidden_states.device
        )
        cos, sin = self.rotary_emb(position_ids, hidden_states.dtype)
        query, key = apply_rotary(query, key, cos, sin, self.rotary_ndims)

        if past_key_value is not None:
            key = torch.cat((past_key_value[0], key), dim=2)
            value = torch.cat((past_key_value[1], value), dim=2)

        # Prefill остаётся dense; sparse route включается только для q_len=1.
        if past_key_value is None or query_length != 1:
            attention_output = self._dense_attention(query, key, value, past_length)
            self.reset_route()
        else:
            attention_output = self._routed_attention(query, key, value)

        attention_output = attention_output.transpose(1, 2).contiguous()
        attention_output = attention_output.view(batch_size, query_length, -1)
        attention_output = self.dense(attention_output)
        present = (key, value) if use_cache else None
        return attention_output, present


def make_ocean_model(dense_model):
    ocean_model = PythiaForCausalLM(dense_model.config)
    for layer_idx, layer in enumerate(ocean_model.gpt_neox.layers):
        layer.attention = OceanPythiaAttention(
            dense_model.config, layer_idx=layer_idx, block_size=64,
            summary_window=100, route_refresh_interval=50,
            local_blocks=2, semantic_blocks=5,
        )
    ocean_model.load_state_dict(dense_model.state_dict(), strict=True)
    ocean_model = ocean_model.to(device=DEVICE, dtype=DTYPE)
    ocean_model.eval()
    return ocean_model


def reset_ocean_routes(ocean_model):
    for layer in ocean_model.gpt_neox.layers:
        layer.attention.reset_route()


ocean_model = make_ocean_model(model)
print('Ocean model created; weights copied from dense model')

Ocean model created; weights copied from dense model


In [14]:
# # Сравнение на 14K prompt. Это benchmark скорости, а не проверка качества
# # за пределами штатного контекста Pythia-1B.
# reset_ocean_routes(ocean_model)
# ocean_benchmark = benchmark_generation(
#     ocean_model, shakespeare_ids, prompt_length=14_000, new_tokens=64,
#     allow_untrained_context=True,
# )

# print(f'prefill seconds: {ocean_benchmark["prefill_seconds"]:.3f}')
# print(f'prefill tok/s: {ocean_benchmark["prefill_tokens_per_second"]:.2f}')
# print(f'Ocean decode seconds: {ocean_benchmark["decode_seconds"]:.3f}')
# print(f'Ocean decode tok/s: {ocean_benchmark["decode_tokens_per_second"]:.2f}')
# print(f'total seconds: {ocean_benchmark["total_seconds"]:.3f}')

# clear_gpu_cache()

prefill seconds: 6.855

prefill tok/s: 2042.38

Ocean decode seconds: 1.009

Ocean decode tok/s: 62.46

total seconds: 7.863

### Ограничения текущего Ocean prototype

1. Prefill пока dense, поэтому оптимизируется только decode.
2. Summary blocks пересчитываются при refresh маршрута полным проходом по KV-cache; hierarchy ещё не добавлена.
3. Реализован только batch=1.
4. Полный KV-cache сохраняется, то есть оптимизация уменьшает вычисления attention, но не память.
5. Для prompt длиннее 2048 качество Pythia не является валидным показателем без отдельного context-extension обучения.

Именно этот вариант предназначен как прозрачная точка старта для дальнейшей замены `_build_route` на иерархический selector и для переноса routing в causal prefill.

## 10. Perplexity: dense vs Ocean

Обычный benchmark perplexity обрабатывает окно целиком и поэтому не включает routed decoding. Здесь каждый токен подаётся отдельно через KV-cache: dense-модель просматривает полный cache, Ocean — выбранные блоки.

Тест ограничен 2048 токенами — штатным контекстом Pythia-1B.

In [15]:
@torch.inference_mode()
def evaluate_cached_perplexity(model, token_ids, max_tokens=2048):
    ids = token_ids[:max_tokens].to(DEVICE)
    if ids.numel() < 2:
        raise ValueError('Для perplexity нужно минимум два токена')

    for layer in model.gpt_neox.layers:
        if hasattr(layer.attention, 'reset_route'):
            layer.attention.reset_route()

    model.eval()
    synchronize()
    start = time.perf_counter()

    # Первый token создаёт KV-cache и logits для следующего token.
    logits, past = model(ids[:1].view(1, 1), use_cache=True)
    total_nll = 0.0
    total_targets = 0

    for index in range(1, ids.numel()):
        target = ids[index].view(1)
        log_probs = F.log_softmax(logits[:, -1, :].float(), dim=-1)
        total_nll += -log_probs[0, target].item()
        total_targets += 1

        if index < ids.numel() - 1:
            logits, past = model(
                ids[index:index + 1].view(1, 1),
                past_key_values=past,
                use_cache=True,
            )

    synchronize()
    elapsed = time.perf_counter() - start
    mean_nll = total_nll / total_targets
    return {
        'mean_nll': mean_nll,
        'perplexity': math.exp(mean_nll),
        'seconds': elapsed,
        'tokens_per_second': total_targets / elapsed,
        'tokens': total_targets,
    }

PPL_TOKENS = min(config.max_position_embeddings, 2048)
dense_ppl = evaluate_cached_perplexity(model, shakespeare_ids, PPL_TOKENS)
ocean_ppl = evaluate_cached_perplexity(ocean_model, shakespeare_ids, PPL_TOKENS)

print('--- dense ---')
print(f'tokens: {dense_ppl["tokens"]}')
print(f'mean NLL: {dense_ppl["mean_nll"]:.4f}')
print(f'perplexity: {dense_ppl["perplexity"]:.2f}')
print(f'time: {dense_ppl["seconds"]:.3f}s')
print(f'tok/s: {dense_ppl["tokens_per_second"]:.2f}')

print('--- Ocean ---')
print(f'tokens: {ocean_ppl["tokens"]}')
print(f'mean NLL: {ocean_ppl["mean_nll"]:.4f}')
print(f'perplexity: {ocean_ppl["perplexity"]:.2f}')
print(f'time: {ocean_ppl["seconds"]:.3f}s')
print(f'tok/s: {ocean_ppl["tokens_per_second"]:.2f}')

print('--- delta ---')
print(f'PPL delta: {ocean_ppl["perplexity"] - dense_ppl["perplexity"]:+.2f}')
print(f'NLL delta: {ocean_ppl["mean_nll"] - dense_ppl["mean_nll"]:+.4f}')
print(f'speedup: {dense_ppl["seconds"] / ocean_ppl["seconds"]:.2f}x')

--- dense ---
tokens: 2047
mean NLL: 3.0701
perplexity: 21.54
time: 28.244s
tok/s: 72.48
--- Ocean ---
tokens: 2047
mean NLL: 3.4758
perplexity: 32.33
time: 31.082s
tok/s: 65.86
--- delta ---
PPL delta: +10.78
NLL delta: +0.4057
speedup: 0.91x


--- dense ---

tokens: 2047

mean NLL: 3.0701

perplexity: 21.54

time: 28.904s

tok/s: 70.82

--- Ocean ---

tokens: 2047

mean NLL: 3.4758

perplexity: 32.33

time: 30.767s

tok/s: 66.53

--- delta ---

PPL delta: +10.78

NLL delta: +0.4057

speedup: 0.94x

## 11. Query-dependent Ocean routing

Предыдущая версия выбирала блоки по среднему последних ключей. Это не учитывает текущий query и объясняет большую потерю качества. Ниже route строится по cosine similarity текущего query к summary каждого KV-блока.

Включён также диагностический режим: он измеряет, какую долю полной dense attention mass покрывают выбранные блоки. Диагностический режим намеренно не используется при измерении скорости.

Все результаты ниже остаются ограничены штатным контекстом Pythia — 2048 токенов.

In [16]:
class QueryOceanPythiaAttention(PythiaAttention):
    def __init__(
        self, config, layer_idx, block_size=64, max_route_blocks=8,
        route_refresh_interval=1, local_blocks=2,
        track_attention_mass=False,
    ):
        super().__init__(config)
        self.layer_idx = layer_idx
        self.block_size = block_size
        self.max_route_blocks = max_route_blocks
        self.route_refresh_interval = route_refresh_interval
        self.local_blocks = local_blocks
        self.track_attention_mass = track_attention_mass
        self.reset_route()
        self.reset_diagnostics()

    def reset_route(self):
        self._route_cache = None
        self._route_num_blocks = None
        self._route_age = 0
        self._route_refresh_count = 0

    def reset_diagnostics(self):
        self._attention_mass_sum = 0.0
        self._attention_mass_count = 0

    @property
    def mean_attention_mass(self):
        if self._attention_mass_count == 0:
            return float('nan')
        return self._attention_mass_sum / self._attention_mass_count

    def _dense_attention(self, query, key, value, past_length):
        mask = self._causal_mask(
            query.shape[2], key.shape[2], past_length, query.device
        )
        return F.scaled_dot_product_attention(
            query, key, value, attn_mask=mask, dropout_p=0.0,
            is_causal=False,
        )

    def _build_route(self, key, query):
        batch_size, num_heads, key_length, _ = key.shape
        if batch_size != 1 or query.shape[2] != 1:
            raise NotImplementedError('QueryOcean поддерживает batch=1 и q_len=1')

        num_blocks = math.ceil(key_length / self.block_size)
        padded_length = num_blocks * self.block_size
        pad = padded_length - key_length
        padded_key = F.pad(key, (0, 0, 0, pad)) if pad else key
        blocks = padded_key.view(
            1, num_heads, num_blocks, self.block_size, self.head_dim
        )
        counts = torch.full(
            (num_blocks,), self.block_size, device=key.device, dtype=key.dtype
        )
        if pad:
            counts[-1] = self.block_size - pad
        summaries = blocks.sum(dim=3) / counts.view(1, 1, num_blocks, 1)

        # Query-dependent score: [batch, heads, blocks].
        query_vector = query[:, :, 0, :].float()
        query_norm = F.normalize(query_vector, dim=-1).unsqueeze(2)
        summary_norm = F.normalize(summaries.float(), dim=-1)
        scores = (query_norm * summary_norm).sum(dim=-1)

        route_count = min(self.max_route_blocks, num_blocks)
        local_count = min(self.local_blocks, route_count)
        local_start = max(0, num_blocks - local_count)
        local_ids = list(range(local_start, num_blocks))
        routes = []
        for head in range(num_heads):
            order = torch.argsort(
                scores[0, head], descending=True, stable=True
            ).tolist()
            selected = local_ids + [
                block_id for block_id in order if block_id not in local_ids
            ][: route_count - local_count]
            routes.append(selected)

        route = torch.tensor(routes, device=key.device, dtype=torch.long)
        self._route_cache = route.unsqueeze(0)
        self._route_num_blocks = num_blocks
        # age=1 means the route has already served one token.
        self._route_age = 1
        self._route_refresh_count += 1
        return self._route_cache

    def _routed_attention(self, query, key, value):
        key_length = key.shape[2]
        num_blocks = math.ceil(key_length / self.block_size)
        must_refresh = (
            self._route_cache is None
            or self._route_age >= self.route_refresh_interval
            or self._route_num_blocks != num_blocks
        )
        route = self._build_route(key, query) if must_refresh else self._route_cache
        if not must_refresh:
            self._route_age += 1

        block_offsets = torch.arange(self.block_size, device=key.device)
        token_ids = route[:, :, :, None] * self.block_size + block_offsets
        valid = token_ids < key_length
        token_ids = token_ids.clamp(max=key_length - 1)
        route_tokens = token_ids.shape[2] * token_ids.shape[3]
        token_ids = token_ids.reshape(1, self.num_attention_heads, route_tokens)
        valid = valid.reshape(1, self.num_attention_heads, route_tokens)

        if self.track_attention_mass:
            dense_scores = torch.matmul(
                query.float(), key.float().transpose(-1, -2)
            ) * self.scale
            dense_probs = dense_scores.softmax(dim=-1)
            selected_mass = dense_probs.gather(
                -1, token_ids.unsqueeze(2)
            ).mul(valid[:, :, None, :]).sum(dim=-1)
            self._attention_mass_sum += selected_mass.mean().item()
            self._attention_mass_count += 1

        gather_ids = token_ids.unsqueeze(-1).expand(
            -1, -1, -1, self.head_dim
        )
        key_route = key.gather(2, gather_ids)
        value_route = value.gather(2, gather_ids)
        route_mask = valid[:, :, None, :]
        return F.scaled_dot_product_attention(
            query, key_route, value_route, attn_mask=route_mask,
            dropout_p=0.0, is_causal=False,
        )

    def forward(self, hidden_states, past_key_value=None, use_cache=False):
        batch_size, query_length, _ = hidden_states.shape
        qkv = self.query_key_value(hidden_states)
        qkv = qkv.view(
            batch_size, query_length, self.num_attention_heads, 3 * self.head_dim
        ).transpose(1, 2)
        query, key, value = qkv.chunk(3, dim=-1)

        past_length = 0 if past_key_value is None else past_key_value[0].shape[2]
        position_ids = torch.arange(
            past_length, past_length + query_length, device=hidden_states.device
        )
        cos, sin = self.rotary_emb(position_ids, hidden_states.dtype)
        query, key = apply_rotary(query, key, cos, sin, self.rotary_ndims)

        if past_key_value is not None:
            key = torch.cat((past_key_value[0], key), dim=2)
            value = torch.cat((past_key_value[1], value), dim=2)

        if past_key_value is None or query_length != 1:
            attention_output = self._dense_attention(query, key, value, past_length)
            self.reset_route()
        else:
            attention_output = self._routed_attention(query, key, value)

        attention_output = attention_output.transpose(1, 2).contiguous()
        attention_output = attention_output.view(batch_size, query_length, -1)
        attention_output = self.dense(attention_output)
        present = (key, value) if use_cache else None
        return attention_output, present


def make_query_ocean_model(
    dense_model, route_blocks=8, route_refresh_interval=1,
    track_attention_mass=False,
):
    routed_model = PythiaForCausalLM(dense_model.config)
    for layer_idx, layer in enumerate(routed_model.gpt_neox.layers):
        layer.attention = QueryOceanPythiaAttention(
            dense_model.config, layer_idx=layer_idx, block_size=64,
            max_route_blocks=route_blocks,
            route_refresh_interval=route_refresh_interval,
            local_blocks=2, track_attention_mass=track_attention_mass,
        )
    routed_model.load_state_dict(dense_model.state_dict(), strict=True)
    routed_model = routed_model.to(device=DEVICE, dtype=DTYPE).eval()
    return routed_model


def reset_query_ocean(model, reset_diagnostics=True):
    for layer in model.gpt_neox.layers:
        layer.attention.reset_route()
        if reset_diagnostics:
            layer.attention.reset_diagnostics()


def configure_query_ocean(model, route_blocks, route_refresh_interval):
    for layer in model.gpt_neox.layers:
        layer.attention.max_route_blocks = route_blocks
        layer.attention.route_refresh_interval = route_refresh_interval
    reset_query_ocean(model)


def evaluate_query_ocean(model, token_ids, max_tokens=2048, track_mass=False):
    for layer in model.gpt_neox.layers:
        layer.attention.track_attention_mass = track_mass
    reset_query_ocean(model)
    result = evaluate_cached_perplexity(model, token_ids, max_tokens=max_tokens)
    if track_mass:
        masses = [
            layer.attention.mean_attention_mass
            for layer in model.gpt_neox.layers
        ]
        result['mean_attention_mass'] = sum(masses) / len(masses)
    return result


query_ocean_model = make_query_ocean_model(
    model, route_blocks=8, route_refresh_interval=1,
    track_attention_mass=False,
)
query_ocean_ppl = evaluate_query_ocean(
    query_ocean_model, shakespeare_ids, max_tokens=2048, track_mass=False
)
print('--- query-dependent Ocean, 8 blocks, refresh=1 ---')
print(f'mean NLL: {query_ocean_ppl["mean_nll"]:.4f}')
print(f'perplexity: {query_ocean_ppl["perplexity"]:.2f}')
print(f'time: {query_ocean_ppl["seconds"]:.3f}s')
print(f'tok/s: {query_ocean_ppl["tokens_per_second"]:.2f}')
print(f'PPL delta vs dense: {query_ocean_ppl["perplexity"] - dense_ppl["perplexity"]:+.2f}')
print(f'speedup vs dense: {dense_ppl["seconds"] / query_ocean_ppl["seconds"]:.2f}x')

--- query-dependent Ocean, 8 blocks, refresh=1 ---
mean NLL: 3.3669
perplexity: 28.99
time: 59.264s
tok/s: 34.54
PPL delta vs dense: +7.44
speedup vs dense: 0.48x


In [17]:
# Диагностика mass recall намеренно выключена по умолчанию: она
# дополнительно считает полный dense softmax и намного медленнее обычного PPL.
RUN_QUERY_MASS_DIAGNOSTIC = False
if RUN_QUERY_MASS_DIAGNOSTIC:
    mass_probe = evaluate_query_ocean(
        query_ocean_model, shakespeare_ids, max_tokens=2048, track_mass=True
    )
    print('--- attention mass diagnostic, full 2048 tokens ---')
    print(f'mean selected dense attention mass: {mass_probe["mean_attention_mass"]:.4f}')

# Небольшой sweep. Полный grid 8/12/16/24/32 x 1/4/16 будет существенно дольше.
RUN_QUERY_ROUTING_SWEEP = False
QUERY_ROUTING_SWEEP = [(8, 1), (12, 1), (16, 1), (24, 4), (32, 16)]

if RUN_QUERY_ROUTING_SWEEP:
    routing_results = []
    for route_blocks, refresh_interval in QUERY_ROUTING_SWEEP:
        configure_query_ocean(query_ocean_model, route_blocks, refresh_interval)
        result = evaluate_query_ocean(
            query_ocean_model, shakespeare_ids, max_tokens=2048, track_mass=False
        )
        row = {
            'route_blocks': route_blocks,
            'refresh_interval': refresh_interval,
            'perplexity': result['perplexity'],
            'mean_nll': result['mean_nll'],
            'tokens_per_second': result['tokens_per_second'],
            'speedup_vs_dense': dense_ppl['seconds'] / result['seconds'],
        }
        routing_results.append(row)
        print(row)


## 12. Улучшенный selector: multi-summary и global blocks

Один средний вектор на 64-токенный блок слишком грубый: разнонаправленные ключи могут взаимно уничтожиться. Эта версия использует несколько summary-векторов внутри блока, всегда сохраняет начало и конец контекста и заполняет оставшийся бюджет query-dependent блоками.

Quality frontier запускается на том же 2048-токенном фрагменте. По умолчанию sweep выключен, поскольку один полный прогон PPL занимает десятки секунд.

In [18]:
class EnhancedQueryOceanAttention(QueryOceanPythiaAttention):
    def __init__(
        self, config, layer_idx, block_size=64, max_route_blocks=8,
        route_refresh_interval=1, local_blocks=2, global_blocks=1,
        summary_parts=4, track_attention_mass=False,
    ):
        super().__init__(
            config, layer_idx=layer_idx, block_size=block_size,
            max_route_blocks=max_route_blocks,
            route_refresh_interval=route_refresh_interval,
            local_blocks=local_blocks,
            track_attention_mass=track_attention_mass,
        )
        if block_size % summary_parts != 0:
            raise ValueError('block_size должен делиться на summary_parts')
        self.global_blocks = global_blocks
        self.summary_parts = summary_parts

    def _build_route(self, key, query):
        batch_size, num_heads, key_length, _ = key.shape
        if batch_size != 1 or query.shape[2] != 1:
            raise NotImplementedError('Enhanced Ocean поддерживает batch=1 и q_len=1')

        num_blocks = math.ceil(key_length / self.block_size)
        padded_length = num_blocks * self.block_size
        pad = padded_length - key_length
        padded_key = F.pad(key, (0, 0, 0, pad)) if pad else key
        part_size = self.block_size // self.summary_parts
        subblocks = padded_key.view(
            1, num_heads, num_blocks, self.summary_parts, part_size, self.head_dim
        )

        counts = torch.full(
            (num_blocks, self.summary_parts), part_size,
            device=key.device, dtype=key.dtype,
        )
        if pad:
            valid_last = key_length - (num_blocks - 1) * self.block_size
            offsets = torch.arange(self.summary_parts, device=key.device) * part_size
            counts[-1] = (valid_last - offsets).clamp(0, part_size)
        safe_counts = counts.clamp_min(1)
        summaries = subblocks.sum(dim=4) / safe_counts.view(
            1, 1, num_blocks, self.summary_parts, 1
        )

        query_norm = F.normalize(
            query[:, :, 0, :].float(), dim=-1
        ).unsqueeze(2).unsqueeze(3)
        summary_norm = F.normalize(summaries.float(), dim=-1)
        part_scores = (query_norm * summary_norm).sum(dim=-1)
        part_scores = part_scores.masked_fill(
            counts.view(1, 1, num_blocks, self.summary_parts) <= 0,
            float('-inf'),
        )
        scores = part_scores.max(dim=-1).values

        route_count = min(self.max_route_blocks, num_blocks)
        local_count = min(self.local_blocks, route_count)
        local_ids = list(range(max(0, num_blocks - local_count), num_blocks))
        global_ids = list(range(min(self.global_blocks, num_blocks)))
        mandatory = local_ids + [
            block_id for block_id in global_ids if block_id not in local_ids
        ][: max(0, route_count - len(local_ids))]

        routes = []
        for head in range(num_heads):
            order = torch.argsort(
                scores[0, head], descending=True, stable=True
            ).tolist()
            selected = mandatory + [
                block_id for block_id in order if block_id not in mandatory
            ][: max(0, route_count - len(mandatory))]
            routes.append(selected)

        route = torch.tensor(routes, device=key.device, dtype=torch.long)
        self._route_cache = route.unsqueeze(0)
        self._route_num_blocks = num_blocks
        self._route_age = 1
        self._route_refresh_count += 1
        return self._route_cache


def make_enhanced_ocean_model(
    dense_model, block_size=64, route_blocks=16, refresh_interval=1,
    global_blocks=1, summary_parts=4, track_attention_mass=False,
):
    routed_model = PythiaForCausalLM(dense_model.config)
    for layer_idx, layer in enumerate(routed_model.gpt_neox.layers):
        layer.attention = EnhancedQueryOceanAttention(
            dense_model.config, layer_idx=layer_idx, block_size=block_size,
            max_route_blocks=route_blocks,
            route_refresh_interval=refresh_interval, local_blocks=2,
            global_blocks=global_blocks, summary_parts=summary_parts,
            track_attention_mass=track_attention_mass,
        )
    routed_model.load_state_dict(dense_model.state_dict(), strict=True)
    return routed_model.to(device=DEVICE, dtype=DTYPE).eval()


def configure_enhanced_ocean(
    model, block_size, route_blocks, refresh_interval,
    global_blocks=1, summary_parts=4,
):
    for layer in model.gpt_neox.layers:
        attention = layer.attention
        attention.block_size = block_size
        attention.max_route_blocks = route_blocks
        attention.route_refresh_interval = refresh_interval
        attention.global_blocks = global_blocks
        attention.summary_parts = summary_parts
    reset_query_ocean(model)


enhanced_ocean_model = make_enhanced_ocean_model(
    model, block_size=64, route_blocks=16, refresh_interval=1,
    global_blocks=1, summary_parts=4, track_attention_mass=False,
)
enhanced_result = evaluate_query_ocean(
    enhanced_ocean_model, shakespeare_ids, max_tokens=2048, track_mass=False
)
print('--- enhanced Ocean, block=64, route=16, summaries=4 ---')
print(f'mean NLL: {enhanced_result["mean_nll"]:.4f}')
print(f'perplexity: {enhanced_result["perplexity"]:.2f}')
print(f'time: {enhanced_result["seconds"]:.3f}s')
print(f'tok/s: {enhanced_result["tokens_per_second"]:.2f}')
print(f'PPL delta vs dense: {enhanced_result["perplexity"] - dense_ppl["perplexity"]:+.2f}')

--- enhanced Ocean, block=64, route=16, summaries=4 ---
mean NLL: 3.0711
perplexity: 21.57
time: 62.357s
tok/s: 32.83
PPL delta vs dense: +0.02


In [20]:
RUN_ENHANCED_QUALITY_FRONTIER = True
ENHANCED_QUALITY_FRONTIER = [
    # block_size, route_blocks, refresh_interval, summary_parts, global_blocks
    (64, 8, 1, 1, 0),
    (64, 8, 1, 4, 1),
    (64, 12, 1, 4, 1),
    (64, 16, 1, 4, 1),
    (64, 24, 1, 4, 1),
    (64, 32, 1, 4, 1),
    (32, 16, 1, 4, 1),
]

if RUN_ENHANCED_QUALITY_FRONTIER:
    enhanced_frontier_results = []
    for block_size, route_blocks, refresh_interval, summary_parts, global_blocks in ENHANCED_QUALITY_FRONTIER:
        configure_enhanced_ocean(
            enhanced_ocean_model, block_size, route_blocks,
            refresh_interval, global_blocks, summary_parts,
        )
        result = evaluate_query_ocean(
            enhanced_ocean_model, shakespeare_ids,
            max_tokens=2048, track_mass=False,
        )
        row = {
            'block_size': block_size,
            'route_blocks': route_blocks,
            'refresh_interval': refresh_interval,
            'summary_parts': summary_parts,
            'global_blocks': global_blocks,
            'perplexity': result['perplexity'],
            'mean_nll': result['mean_nll'],
            'tokens_per_second': result['tokens_per_second'],
            'speedup_vs_dense': dense_ppl['seconds'] / result['seconds'],
        }
        enhanced_frontier_results.append(row)
        print(row)


{'block_size': 64, 'route_blocks': 8, 'refresh_interval': 1, 'summary_parts': 1, 'global_blocks': 0, 'perplexity': 28.98838293475302, 'mean_nll': 3.3668951612726916, 'tokens_per_second': 32.52692939534521, 'speedup_vs_dense': 0.4487927000723217}
{'block_size': 64, 'route_blocks': 8, 'refresh_interval': 1, 'summary_parts': 4, 'global_blocks': 1, 'perplexity': 21.870551191366623, 'mean_nll': 3.085141037451405, 'tokens_per_second': 32.752554278037664, 'speedup_vs_dense': 0.45190577598171067}
{'block_size': 64, 'route_blocks': 12, 'refresh_interval': 1, 'summary_parts': 4, 'global_blocks': 1, 'perplexity': 21.65760081255577, 'mean_nll': 3.0753564696278763, 'tokens_per_second': 32.81512290586192, 'speedup_vs_dense': 0.45276907122485427}
{'block_size': 64, 'route_blocks': 16, 'refresh_interval': 1, 'summary_parts': 4, 'global_blocks': 1, 'perplexity': 21.56557548359742, 'mean_nll': 3.0710983157440856, 'tokens_per_second': 32.53758654743251, 'speedup_vs_dense': 0.44893974291187805}
{'block_si

## 13. Attention-only microbenchmark

Этот benchmark отделяет attention от MLP и остальной модели. Измеряются два уровня:

1. чистое attention-ядро на заранее подготовленных `Q/K/V`;
2. полный attention-модуль одного слоя, включая QKV/output projections, routing и gather, но без MLP.

Контекст 14K здесь используется только как performance stress test. Это не benchmark качества Pythia.

In [29]:
def _benchmark_seconds(fn, warmup=10, repeats=50):
    for _ in range(warmup):
        fn()
    synchronize()
    start = time.perf_counter()
    for _ in range(repeats):
        fn()
    synchronize()
    return (time.perf_counter() - start) / repeats


def _make_last_block_route_kv(key, value, route_blocks, block_size=64):
    # Для kernel benchmark заранее выбираем последние полные блоки.
    # Стоимость выбора route сюда не входит.
    key_length = key.shape[2]
    route_tokens = min(key_length, route_blocks * block_size)
    return key[:, :, -route_tokens:, :].contiguous(), value[:, :, -route_tokens:, :].contiguous()


@torch.inference_mode()
def benchmark_attention_only(
    model, routed_model, context_lengths=(512, 2048, 4096, 8192, 14000),
    route_blocks_list=(8, 16, 24), block_size=64,
):
    dense_attention = model.gpt_neox.layers[0].attention
    routed_attention = routed_model.gpt_neox.layers[0].attention
    results = []

    for context_length in context_lengths:
        query = torch.randn(
            1, config.num_attention_heads, 1, config.head_dim,
            device=DEVICE, dtype=DTYPE,
        )
        key = torch.randn(
            1, config.num_attention_heads, context_length, config.head_dim,
            device=DEVICE, dtype=DTYPE,
        )
        value = torch.randn_like(key)
        repeats = 50 if context_length <= 2048 else 20

        dense_seconds = _benchmark_seconds(
            lambda: F.scaled_dot_product_attention(
                query, key, value, dropout_p=0.0, is_causal=False
            ),
            warmup=10, repeats=repeats,
        )

        for route_blocks in route_blocks_list:
            key_route, value_route = _make_last_block_route_kv(
                key, value, route_blocks, block_size
            )
            routed_seconds = _benchmark_seconds(
                lambda: F.scaled_dot_product_attention(
                    query, key_route, value_route,
                    dropout_p=0.0, is_causal=False,
                ),
                warmup=10, repeats=repeats,
            )
            results.append({
                'level': 'attention_kernel',
                'context': context_length,
                'route_blocks': route_blocks,
                'dense_ms': dense_seconds * 1000,
                'ocean_ms': routed_seconds * 1000,
                'speedup': dense_seconds / routed_seconds,
                'selected_tokens': key_route.shape[2],
            })
    return results


@torch.inference_mode()
def benchmark_attention_modules(
    model, routed_model, context_lengths=(512, 2048, 4096, 8192, 14000),
    route_blocks_list=(8, 16, 24), block_size=64,
):
    dense_attention = model.gpt_neox.layers[0].attention
    routed_attention = routed_model.gpt_neox.layers[0].attention
    hidden_size = config.hidden_size
    results = []

    for context_length in context_lengths:
        hidden_states = torch.randn(
            1, 1, hidden_size, device=DEVICE, dtype=DTYPE
        )
        past_key = torch.randn(
            1, config.num_attention_heads, context_length, config.head_dim,
            device=DEVICE, dtype=DTYPE,
        )
        past_value = torch.randn_like(past_key)
        past = (past_key, past_value)
        repeats = 30 if context_length <= 2048 else 10

        dense_attention.eval()
        dense_seconds = _benchmark_seconds(
            lambda: dense_attention(
                hidden_states, past_key_value=past, use_cache=False
            ),
            warmup=5, repeats=repeats,
        )

        for route_blocks in route_blocks_list:
            configure_enhanced_ocean(
                routed_model, block_size, route_blocks,
                refresh_interval=1, global_blocks=1, summary_parts=4,
            )
            routed_attention.eval()
            routed_seconds = _benchmark_seconds(
                lambda: routed_attention(
                    hidden_states, past_key_value=past, use_cache=False
                ),
                warmup=5, repeats=repeats,
            )
            results.append({
                'level': 'attention_module',
                'context': context_length,
                'route_blocks': route_blocks,
                'dense_ms': dense_seconds * 1000,
                'ocean_ms': routed_seconds * 1000,
                'speedup': dense_seconds / routed_seconds,
                'selected_tokens': min(context_length, route_blocks * block_size),
            })
    return results


RUN_ATTENTION_MICROBENCHMARK = True
if RUN_ATTENTION_MICROBENCHMARK:
    attention_kernel_results = benchmark_attention_only(
        model, enhanced_ocean_model,
    )
    attention_module_results = benchmark_attention_modules(
        model, enhanced_ocean_model,
    )

    print('--- attention kernel only ---')
    for row in attention_kernel_results:
        print(row)
    print('--- one attention module ---')
    for row in attention_module_results:
        print(row)


--- attention kernel only ---
{'level': 'attention_kernel', 'context': 512, 'route_blocks': 8, 'dense_ms': 0.21745488047599792, 'ocean_ms': 0.2262318041175604, 'speedup': 0.961203847196473, 'selected_tokens': 512}
{'level': 'attention_kernel', 'context': 512, 'route_blocks': 16, 'dense_ms': 0.21745488047599792, 'ocean_ms': 0.21121980156749487, 'speedup': 1.029519386261286, 'selected_tokens': 512}
{'level': 'attention_kernel', 'context': 512, 'route_blocks': 24, 'dense_ms': 0.21745488047599792, 'ocean_ms': 0.19994453992694616, 'speedup': 1.0875759875986086, 'selected_tokens': 512}
{'level': 'attention_kernel', 'context': 2048, 'route_blocks': 8, 'dense_ms': 0.1967302616685629, 'ocean_ms': 0.1822296204045415, 'speedup': 1.0795734591985136, 'selected_tokens': 512}
{'level': 'attention_kernel', 'context': 2048, 'route_blocks': 16, 'dense_ms': 0.1967302616685629, 'ocean_ms': 0.18657677806913853, 'speedup': 1.0544198678126055, 'selected_tokens': 1024}
{'level': 'attention_kernel', 'context':

## 14. Decomposition of Ocean overhead

Следующий benchmark разделяет стоимость sparse attention на компоненты:

- dense SDPA;
- Ocean SDPA на заранее contiguous route;
- реальный `gather + SDPA` с разрозненным route;
- построение summaries и route;
- расширение KV-cache через текущий `torch.cat`.

Все времена относятся к одному attention head-group и не включают MLP.

In [36]:
def _make_fixed_route_indices(
    context_length, route_blocks, block_size, num_heads, device
):
    num_blocks = math.ceil(context_length / block_size)
    route_count = min(route_blocks, num_blocks)
    selected = []
    candidates = [0] + list(range(max(0, num_blocks - 2), num_blocks))
    candidates += list(range(num_blocks))
    for block_id in candidates:
        if block_id not in selected:
            selected.append(block_id)
        if len(selected) == route_count:
            break

    selected = torch.tensor(selected, device=device, dtype=torch.long)
    offsets = torch.arange(block_size, device=device)
    token_ids = selected[None, None, :, None] * block_size + offsets
    valid = token_ids < context_length
    token_ids = token_ids.clamp(max=context_length - 1)
    route_tokens = token_ids.shape[2] * token_ids.shape[3]
    token_ids = token_ids.reshape(1, 1, route_tokens).expand(
        1, num_heads, route_tokens
    )
    valid = valid.reshape(1, 1, route_tokens).expand(
        1, num_heads, route_tokens
    )
    return token_ids, valid


@torch.inference_mode()
def benchmark_attention_decomposition(
    model, routed_model, context_lengths=(512, 2048, 4096, 8192, 14000),
    route_blocks_list=(8, 16, 24), block_size=64,
):
    routed_attention = routed_model.gpt_neox.layers[0].attention
    results = []

    for context_length in context_lengths:
        query = torch.randn(
            1, config.num_attention_heads, 1, config.head_dim,
            device=DEVICE, dtype=DTYPE,
        )
        key = torch.randn(
            1, config.num_attention_heads, context_length, config.head_dim,
            device=DEVICE, dtype=DTYPE,
        )
        value = torch.randn_like(key)
        new_key = torch.randn(
            1, config.num_attention_heads, 1, config.head_dim,
            device=DEVICE, dtype=DTYPE,
        )
        new_value = torch.randn_like(new_key)
        repeats = 20 if context_length <= 2048 else 8

        dense_seconds = _benchmark_seconds(
            lambda: F.scaled_dot_product_attention(
                query, key, value, dropout_p=0.0, is_causal=False
            ),
            warmup=5, repeats=repeats,
        )

        for route_blocks in route_blocks_list:
            token_ids, valid = _make_fixed_route_indices(
                context_length, route_blocks, block_size,
                config.num_attention_heads, DEVICE,
            )
            gather_ids = token_ids.unsqueeze(-1).expand(
                -1, -1, -1, config.head_dim
            )
            route_mask = valid[:, :, None, :]
            key_route = key.gather(2, gather_ids).contiguous()
            value_route = value.gather(2, gather_ids).contiguous()
            selected_tokens = int(valid.sum().item()) // config.num_attention_heads

            contiguous_seconds = _benchmark_seconds(
                lambda: F.scaled_dot_product_attention(
                    query, key_route, value_route,
                    dropout_p=0.0, is_causal=False,
                ),
                warmup=5, repeats=repeats,
            )

            gather_seconds = _benchmark_seconds(
                lambda: F.scaled_dot_product_attention(
                    query, key.gather(2, gather_ids),
                    value.gather(2, gather_ids),
                    attn_mask=route_mask, dropout_p=0.0, is_causal=False,
                ),
                warmup=5, repeats=repeats,
            )

            configure_enhanced_ocean(
                routed_model, block_size, route_blocks,
                refresh_interval=1, global_blocks=1, summary_parts=4,
            )
            route_build_seconds = _benchmark_seconds(
                lambda: routed_attention._build_route(key, query),
                warmup=3, repeats=max(3, repeats // 2),
            )
            cache_append_seconds = _benchmark_seconds(
                lambda: (
                    torch.cat((key, new_key), dim=2),
                    torch.cat((value, new_value), dim=2),
                ),
                warmup=3, repeats=max(3, repeats // 2),
            )

            results.append({
                'context': context_length,
                'route_blocks': route_blocks,
                'selected_tokens': selected_tokens,
                'dense_core_ms': dense_seconds * 1000,
                'contiguous_ocean_core_ms': contiguous_seconds * 1000,
                'gather_plus_attention_ms': gather_seconds * 1000,
                'route_build_ms': route_build_seconds * 1000,
                'kv_append_ms': cache_append_seconds * 1000,
                'ideal_core_speedup': dense_seconds / contiguous_seconds,
                'gather_core_speedup': dense_seconds / gather_seconds,
            })
    return results


RUN_ATTENTION_DECOMPOSITION = True
if RUN_ATTENTION_DECOMPOSITION:
    attention_decomposition_results = benchmark_attention_decomposition(
        model, enhanced_ocean_model,
    )
    for row in attention_decomposition_results:
        print(row)


{'context': 512, 'route_blocks': 8, 'selected_tokens': 512, 'dense_core_ms': 0.21791469771414995, 'contiguous_ocean_core_ms': 0.261355796828866, 'gather_plus_attention_ms': 0.34943469800055027, 'route_build_ms': 1.0477468837052584, 'kv_append_ms': 0.018610595725476742, 'ideal_core_speedup': 0.8337855917419692, 'gather_core_speedup': 0.623620662060889}
{'context': 512, 'route_blocks': 16, 'selected_tokens': 512, 'dense_core_ms': 0.21791469771414995, 'contiguous_ocean_core_ms': 0.18466764595359564, 'gather_plus_attention_ms': 0.2469760016538203, 'route_build_ms': 0.9070119122043252, 'kv_append_ms': 0.018693995662033558, 'ideal_core_speedup': 1.180037231691949, 'gather_core_speedup': 0.8823314664377602}
{'context': 512, 'route_blocks': 24, 'selected_tokens': 512, 'dense_core_ms': 0.21791469771414995, 'contiguous_ocean_core_ms': 0.22765524918213487, 'gather_plus_attention_ms': 0.30117150163277984, 'route_build_ms': 1.0152481030672789, 'kv_append_ms': 0.018456880934536457, 'ideal_core_speed

## 15. Incremental route optimization

В предыдущей реализации summaries пересчитывались полным проходом по KV-cache при каждом refresh, а semantic-блоки выбирались через Python `argsort`. Ниже summaries обновляются добавлением только нового ключа, а выбор блоков выполняется через GPU `topk`.

Первый decode token после prefill всё ещё требует инициализации summaries по существующему cache. После этого обновление summaries имеет стоимость `O(D)` на новый token; score всех block summaries считается только при refresh.

In [51]:
class IncrementalOceanAttention(EnhancedQueryOceanAttention):
    def reset_route(self):
        super().reset_route()
        self._summary_sums = None
        self._summary_counts = None

    def _initialize_summary_cache(self, key):
        _, num_heads, key_length, _ = key.shape
        num_blocks = math.ceil(key_length / self.block_size)
        padded_length = num_blocks * self.block_size
        pad = padded_length - key_length
        padded_key = F.pad(key, (0, 0, 0, pad)) if pad else key
        part_size = self.block_size // self.summary_parts
        subblocks = padded_key.view(
            1, num_heads, num_blocks, self.summary_parts, part_size, self.head_dim
        )
        counts = torch.full(
            (num_blocks, self.summary_parts), part_size,
            device=key.device, dtype=torch.int32,
        )
        if pad:
            valid_last = key_length - (num_blocks - 1) * self.block_size
            offsets = torch.arange(self.summary_parts, device=key.device) * part_size
            counts[-1] = (valid_last - offsets).clamp(0, part_size)
        self._summary_sums = subblocks.sum(dim=4)
        self._summary_counts = counts

    def _append_summary_token(self, new_key, absolute_position):
        block_id = absolute_position // self.block_size
        part_id = (absolute_position % self.block_size) // (
            self.block_size // self.summary_parts
        )
        current_blocks = self._summary_sums.shape[2]
        if block_id >= current_blocks:
            zeros = torch.zeros(
                1, self.num_attention_heads, 1, self.summary_parts, self.head_dim,
                device=new_key.device, dtype=new_key.dtype,
            )
            count_zeros = torch.zeros(
                1, self.summary_parts, device=new_key.device, dtype=torch.int32
            )
            self._summary_sums = torch.cat((self._summary_sums, zeros), dim=2)
            self._summary_counts = torch.cat((self._summary_counts, count_zeros), dim=0)
        self._summary_sums[:, :, block_id, part_id, :] += new_key[:, :, 0, :]
        self._summary_counts[block_id, part_id] += 1

    def _update_summary_cache(self, full_key, new_key, past_length):
        if self._summary_sums is None:
            self._initialize_summary_cache(full_key)
        else:
            expected_blocks = math.ceil(full_key.shape[2] / self.block_size)
            if self._summary_sums.shape[2] > expected_blocks:
                self._initialize_summary_cache(full_key)
            else:
                self._append_summary_token(new_key, past_length)

    def _summary_scores(self, query):
        safe_counts = self._summary_counts.clamp_min(1).to(
            dtype=self._summary_sums.dtype
        )
        summaries = self._summary_sums / safe_counts.view(
            1, 1, safe_counts.shape[0], safe_counts.shape[1], 1
        )
        query_norm = F.normalize(
            query[:, :, 0, :].float(), dim=-1
        ).unsqueeze(2).unsqueeze(3)
        summary_norm = F.normalize(summaries.float(), dim=-1)
        part_scores = (query_norm * summary_norm).sum(dim=-1)
        part_scores = part_scores.masked_fill(
            self._summary_counts.view(1, 1, safe_counts.shape[0], safe_counts.shape[1]) <= 0,
            float('-inf'),
        )
        return part_scores.max(dim=-1).values

    def _build_incremental_route(self, query):
        scores = self._summary_scores(query)
        num_blocks = scores.shape[-1]
        route_count = min(self.max_route_blocks, num_blocks)
        local_count = min(self.local_blocks, route_count)
        global_count = min(
            self.global_blocks, max(0, route_count - local_count)
        )
        local_ids = list(range(num_blocks - local_count, num_blocks))
        global_ids = list(range(global_count))
        mandatory_ids = global_ids + [
            block_id for block_id in local_ids if block_id not in global_ids
        ]
        mandatory = torch.tensor(
            mandatory_ids, device=query.device, dtype=torch.long
        )
        semantic_count = route_count - len(mandatory_ids)
        masked_scores = scores.clone()
        if mandatory.numel():
            masked_scores[:, :, mandatory] = float('-inf')
        if semantic_count > 0:
            semantic = torch.topk(
                masked_scores, k=semantic_count, dim=-1, largest=True, sorted=True
            ).indices
            mandatory = mandatory.view(1, 1, -1).expand(
                1, self.num_attention_heads, -1
            )
            route = torch.cat((mandatory, semantic), dim=-1)
        else:
            route = mandatory.view(1, 1, -1).expand(
                1, self.num_attention_heads, -1
            )
        self._route_cache = route
        self._route_num_blocks = num_blocks
        self._route_age = 1
        self._route_refresh_count += 1
        return route

    def _routed_attention_incremental(
        self, query, key, value, new_key, past_length
    ):
        self._update_summary_cache(key, new_key, past_length)
        num_blocks = self._summary_sums.shape[2]
        must_refresh = (
            self._route_cache is None
            or self._route_age >= self.route_refresh_interval
            or self._route_num_blocks != num_blocks
        )
        route = (
            self._build_incremental_route(query)
            if must_refresh else self._route_cache
        )
        if not must_refresh:
            self._route_age += 1

        block_offsets = torch.arange(self.block_size, device=key.device)
        token_ids = route[:, :, :, None] * self.block_size + block_offsets
        valid = token_ids < key.shape[2]
        token_ids = token_ids.clamp(max=key.shape[2] - 1)
        route_tokens = token_ids.shape[2] * token_ids.shape[3]
        token_ids = token_ids.reshape(1, self.num_attention_heads, route_tokens)
        valid = valid.reshape(1, self.num_attention_heads, route_tokens)
        gather_ids = token_ids.unsqueeze(-1).expand(
            -1, -1, -1, self.head_dim
        )
        key_route = key.gather(2, gather_ids)
        value_route = value.gather(2, gather_ids)
        return F.scaled_dot_product_attention(
            query, key_route, value_route,
            attn_mask=valid[:, :, None, :], dropout_p=0.0, is_causal=False
        )

    def forward(self, hidden_states, past_key_value=None, use_cache=False):
        batch_size, query_length, _ = hidden_states.shape
        qkv = self.query_key_value(hidden_states)
        qkv = qkv.view(
            batch_size, query_length, self.num_attention_heads, 3 * self.head_dim
        ).transpose(1, 2)
        query, new_key, value = qkv.chunk(3, dim=-1)
        past_length = 0 if past_key_value is None else past_key_value[0].shape[2]
        position_ids = torch.arange(
            past_length, past_length + query_length, device=hidden_states.device
        )
        cos, sin = self.rotary_emb(position_ids, hidden_states.dtype)
        query, new_key = apply_rotary(
            query, new_key, cos, sin, self.rotary_ndims
        )
        key = new_key
        if past_key_value is not None:
            key = torch.cat((past_key_value[0], new_key), dim=2)
            value = torch.cat((past_key_value[1], value), dim=2)

        if past_key_value is None or query_length != 1:
            attention_output = self._dense_attention(query, key, value, past_length)
            self.reset_route()
        else:
            attention_output = self._routed_attention_incremental(
                query, key, value, new_key, past_length
            )

        attention_output = attention_output.transpose(1, 2).contiguous()
        attention_output = attention_output.view(batch_size, query_length, -1)
        attention_output = self.dense(attention_output)
        present = (key, value) if use_cache else None
        return attention_output, present


def make_incremental_ocean_model(
    dense_model, route_blocks=16, refresh_interval=1,
    block_size=64, summary_parts=4, global_blocks=1,
):
    routed_model = PythiaForCausalLM(dense_model.config)
    for layer_idx, layer in enumerate(routed_model.gpt_neox.layers):
        layer.attention = IncrementalOceanAttention(
            dense_model.config, layer_idx=layer_idx, block_size=block_size,
            max_route_blocks=route_blocks,
            route_refresh_interval=refresh_interval, local_blocks=2,
            global_blocks=global_blocks, summary_parts=summary_parts,
        )
    routed_model.load_state_dict(dense_model.state_dict(), strict=True)
    return routed_model.to(device=DEVICE, dtype=DTYPE).eval()


def configure_incremental_ocean(model, route_blocks, refresh_interval):
    for layer in model.gpt_neox.layers:
        layer.attention.max_route_blocks = route_blocks
        layer.attention.route_refresh_interval = refresh_interval
    for layer in model.gpt_neox.layers:
        layer.attention.reset_route()


def evaluate_incremental_ocean(model, token_ids, max_tokens=2048):
    for layer in model.gpt_neox.layers:
        layer.attention.reset_route()
    result = evaluate_cached_perplexity(model, token_ids, max_tokens=max_tokens)
    result['route_refreshes'] = sum(
        layer.attention._route_refresh_count
        for layer in model.gpt_neox.layers
    )
    return result

In [ ]:

incremental_ocean_model = make_incremental_ocean_model(
    model, route_blocks=16, refresh_interval=1,
    block_size=64, summary_parts=4, global_blocks=1,
)

RUN_INCREMENTAL_ROUTE_BENCHMARK = True
if RUN_INCREMENTAL_ROUTE_BENCHMARK:
    incremental_route_results = []
    for refresh_interval in (1, 4, 16):
        configure_incremental_ocean(
            incremental_ocean_model, route_blocks=16,
            refresh_interval=refresh_interval,
        )
        result = evaluate_incremental_ocean(
            incremental_ocean_model, shakespeare_ids, max_tokens=2048
        )
        row = {
            'route_blocks': 16,
            'refresh_interval': refresh_interval,
            'perplexity': result['perplexity'],
            'mean_nll': result['mean_nll'],
            'seconds': result['seconds'],
            'tokens_per_second': result['tokens_per_second'],
            'speedup_vs_dense': dense_ppl['seconds'] / result['seconds'],
            'route_refreshes_all_layers': result['route_refreshes'],
        }
        incremental_route_results.append(row)
        print(row)

## 16. 14K decode benchmark for incremental Ocean

Этот тест измеряет только производительность длинного KV-cache. Контекст 14K выходит за пределы обученного контекста Pythia-1B, поэтому здесь не считаются perplexity или качество генерации.

Сравниваются dense и incremental Ocean с route budget 16 и refresh interval 4/16. Prefill остаётся dense; ожидаемый выигрыш должен проявиться только в decode.

In [1]:
clear_gpu_cache()
RUN_INCREMENTAL_14K_BENCHMARK = True
LONG_PROMPT_LENGTH = 14_000
LONG_NEW_TOKENS = 64

if RUN_INCREMENTAL_14K_BENCHMARK:
    dense_long_benchmark = benchmark_generation(
        model, shakespeare_ids,
        prompt_length=LONG_PROMPT_LENGTH, new_tokens=LONG_NEW_TOKENS,
        allow_untrained_context=True,
    )
    print('--- dense 14K ---')
    print(f'prefill seconds: {dense_long_benchmark["prefill_seconds"]:.3f}')
    print(f'prefill tok/s: {dense_long_benchmark["prefill_tokens_per_second"]:.2f}')
    print(f'decode seconds: {dense_long_benchmark["decode_seconds"]:.3f}')
    print(f'decode tok/s: {dense_long_benchmark["decode_tokens_per_second"]:.2f}')
    print(f'total seconds: {dense_long_benchmark["total_seconds"]:.3f}')

    incremental_14k_results = []
    for refresh_interval in (4, 16):
        configure_incremental_ocean(
            incremental_ocean_model, route_blocks=16,
            refresh_interval=refresh_interval,
        )
        ocean_long_benchmark = benchmark_generation(
            incremental_ocean_model, shakespeare_ids,
            prompt_length=LONG_PROMPT_LENGTH, new_tokens=LONG_NEW_TOKENS,
            allow_untrained_context=True,
        )
        row = {
            'refresh_interval': refresh_interval,
            'prefill_seconds': ocean_long_benchmark['prefill_seconds'],
            'decode_seconds': ocean_long_benchmark['decode_seconds'],
            'decode_tok_s': ocean_long_benchmark['decode_tokens_per_second'],
            'total_seconds': ocean_long_benchmark['total_seconds'],
            'prefill_speedup': dense_long_benchmark['prefill_seconds'] / ocean_long_benchmark['prefill_seconds'],
            'decode_speedup': dense_long_benchmark['decode_seconds'] / ocean_long_benchmark['decode_seconds'],
            'total_speedup': dense_long_benchmark['total_seconds'] / ocean_long_benchmark['total_seconds'],
        }
        incremental_14k_results.append(row)
        print('--- incremental Ocean 14K ---')
        print(row)


NameError: name 'clear_gpu_cache' is not defined

## 17. Sublinear streaming path: preallocated cache + hierarchical routing

Эта экспериментальная реализация убирает два линейных bottleneck предыдущей версии: `torch.cat` для KV-cache и полный scan block summaries при каждом route refresh. KV-cache и summary tree выделяются заранее. При добавлении токена обновляется только путь от leaf до root, а route выбирается beam search по дереву.

Prefill также выполняется causal token-by-token через sparse attention. Это даёт линейную асимптотику по числу токенов при фиксированных `route_blocks`, `beam_width` и размерности модели, но Python-level streaming может быть медленнее dense fused prefill.

In [16]:
class StaticHierarchicalKVCache:
    def __init__(self, config, max_length, block_size, summary_parts, device, dtype):
        self.max_length = max_length
        self.block_size = block_size
        self.summary_parts = summary_parts
        self.part_size = block_size // summary_parts
        self.num_heads = config.num_attention_heads
        self.head_dim = config.head_dim
        self.key = torch.empty(
            1, self.num_heads, max_length, self.head_dim,
            device=device, dtype=dtype,
        )
        self.value = torch.empty_like(self.key)
        max_blocks = math.ceil(max_length / block_size)
        tree_capacity = 1
        while tree_capacity < max_blocks:
            tree_capacity *= 2
        self.tree_capacity = tree_capacity
        node_count = 2 * tree_capacity - 1
        self.leaf_start = tree_capacity - 1
        # Head-first layout avoids a permute before every route score.
        self.tree_sums = torch.zeros(
            self.num_heads, node_count, summary_parts, self.head_dim,
            device=device, dtype=dtype,
        )
        self.tree_counts = torch.zeros(
            node_count, summary_parts, device=device, dtype=torch.int32
        )
        # Reused by every token/layer instead of allocating in forward_token.
        self.offsets = torch.arange(block_size, device=device, dtype=torch.long)
        self.position_ids = torch.arange(max_length, device=device, dtype=torch.long)


class SublinearOceanAttention(PythiaAttention):
    def __init__(
        self, config, layer_idx, block_size=64, route_blocks=16,
        beam_width=32, summary_parts=4, global_blocks=1, local_blocks=2,
        route_refresh_interval=1,
    ):
        super().__init__(config)
        if block_size % summary_parts != 0:
            raise ValueError('block_size должен делиться на summary_parts')
        self.layer_idx = layer_idx
        self.block_size = block_size
        self.max_route_blocks = route_blocks
        self.beam_width = beam_width
        self.summary_parts = summary_parts
        self.global_blocks = global_blocks
        self.local_blocks = local_blocks
        self.route_refresh_interval = route_refresh_interval
        self.reset_route()
    def reset_route(self):
        self._route_cache = None
        self._route_age = 0
        self._route_num_blocks = -1

    @torch.no_grad()
    def _update_hierarchy(self, cache, new_key, position):
        block_id = position // cache.block_size
        part_id = (position % cache.block_size) // cache.part_size
        node = cache.leaf_start + block_id
        delta = new_key[0, :, 0, :]
        while True:
            cache.tree_sums[:, node, part_id, :].add_(delta)
            cache.tree_counts[node, part_id].add_(1)
            if node == 0:
                break
            node = (node - 1) // 2

    def _node_scores(self, query_vector, cache, node_ids):
        # node_ids: [heads, candidates]; result: [heads, candidates].
        gather_ids = node_ids[:, :, None, None].expand(
            node_ids.shape[0], node_ids.shape[1],
            cache.summary_parts, cache.head_dim
        )
        sums = cache.tree_sums.gather(1, gather_ids)
        counts = cache.tree_counts[node_ids]
        safe_counts = counts.clamp_min(1).to(dtype=sums.dtype)
        summaries = sums / safe_counts.unsqueeze(-1)
        query_norm = F.normalize(query_vector.float(), dim=-1).unsqueeze(1).unsqueeze(2)
        summary_norm = F.normalize(summaries.float(), dim=-1)
        part_scores = (query_norm * summary_norm).sum(dim=-1)
        part_scores = part_scores.masked_fill(counts <= 0, float('-inf'))
        return part_scores.max(dim=-1).values

    @torch.no_grad()
    def _select_route(self, query, cache, length):
        num_blocks = math.ceil(length / cache.block_size)
        route_count = min(self.max_route_blocks, num_blocks)
        local_count = min(self.local_blocks, route_count)
        global_count = min(self.global_blocks, max(0, route_count - local_count))
        device = query.device
        global_ids = torch.arange(global_count, device=device, dtype=torch.long)
        local_ids = torch.arange(
            num_blocks - local_count, num_blocks,
            device=device, dtype=torch.long,
        )
        mandatory_ids = torch.cat((global_ids, local_ids), dim=0)
        semantic_count = route_count - mandatory_ids.numel()

        heads = self.num_attention_heads
        query_vector = query[:, :, 0, :].reshape(heads, self.head_dim)
        beam = min(max(self.beam_width, route_count), num_blocks)
        candidates = torch.zeros(
            heads, 1, device=query.device, dtype=torch.long
        )
        levels = int(math.log2(cache.tree_capacity))
        for _ in range(levels):
            left = candidates * 2 + 1
            right = left + 1
            children = torch.cat((left, right), dim=-1)
            scores = self._node_scores(query_vector, cache, children)
            keep = min(beam, children.shape[-1])
            order = torch.topk(scores, k=keep, dim=-1, largest=True, sorted=True).indices
            candidates = children.gather(-1, order)

        leaf_ids = candidates - cache.leaf_start
        if semantic_count == 0:
            route = mandatory_ids.view(1, -1).expand(heads, -1)
        else:
            # Remove mandatory blocks without .tolist(), CPU sync, or Python lists.
            mandatory_mask = (
                leaf_ids.unsqueeze(-1) == mandatory_ids.view(1, 1, -1)
            ).any(dim=-1)
            valid_rank = torch.arange(beam, device=device).view(1, -1)
            valid_rank = valid_rank.expand(heads, -1).masked_fill(
                mandatory_mask, beam
            )
            order = valid_rank.argsort(dim=-1, stable=True)
            semantic_ids = leaf_ids.gather(1, order[:, :semantic_count])
            route = torch.cat(
                (mandatory_ids.view(1, -1).expand(heads, -1), semantic_ids),
                dim=-1,
            )
        return route.unsqueeze(0)

    @torch.no_grad()
    def forward_token(self, hidden_states, cache, position):
        batch_size, query_length, _ = hidden_states.shape
        if batch_size != 1 or query_length != 1:
            raise NotImplementedError('Sublinear path поддерживает только batch=1, q_len=1')
        qkv = self.query_key_value(hidden_states)
        qkv = qkv.view(1, 1, self.num_attention_heads, 3 * self.head_dim).transpose(1, 2)
        query, new_key, new_value = qkv.chunk(3, dim=-1)
        position_ids = cache.position_ids[position:position + 1]
        cos, sin = self.rotary_emb(position_ids, hidden_states.dtype)
        query, new_key = apply_rotary(query, new_key, cos, sin, self.rotary_ndims)

        cache.key[:, :, position:position + 1, :].copy_(new_key)
        cache.value[:, :, position:position + 1, :].copy_(new_value)
        self._update_hierarchy(cache, new_key, position)
        length = position + 1
        num_blocks = math.ceil(length / cache.block_size)
        must_refresh = (
            self._route_cache is None
            or self._route_age >= self.route_refresh_interval
            or self._route_num_blocks != num_blocks
        )
        if must_refresh:
            route = self._select_route(query, cache, length)
            self._route_cache = route
            self._route_age = 0
            self._route_num_blocks = num_blocks
        else:
            route = self._route_cache
            self._route_age += 1

        offsets = cache.offsets
        token_ids = route[:, :, :, None] * self.block_size + offsets
        valid = token_ids < length
        token_ids = token_ids.clamp(max=length - 1)
        route_tokens = token_ids.shape[2] * token_ids.shape[3]
        token_ids = token_ids.reshape(1, self.num_attention_heads, route_tokens)
        valid = valid.reshape(1, self.num_attention_heads, route_tokens)
        gather_ids = token_ids.unsqueeze(-1).expand(-1, -1, -1, self.head_dim)
        key_route = cache.key.gather(2, gather_ids)
        value_route = cache.value.gather(2, gather_ids)
        attention_output = F.scaled_dot_product_attention(
            query, key_route, value_route,
            attn_mask=valid[:, :, None, :], dropout_p=0.0, is_causal=False
        )
        attention_output = attention_output.transpose(1, 2).contiguous()
        attention_output = attention_output.view(1, 1, -1)
        return self.dense(attention_output)


class SublinearPythiaForCausalLM(PythiaForCausalLM):
    def __init__(
        self, config, block_size=64, route_blocks=16, beam_width=32,
        summary_parts=4, global_blocks=1, local_blocks=2,
        route_refresh_interval=1,
    ):
        super().__init__(config)
        self.sublinear_kwargs = {
            'block_size': block_size, 'route_blocks': route_blocks,
            'beam_width': beam_width, 'summary_parts': summary_parts,
            'global_blocks': global_blocks, 'local_blocks': local_blocks,
            'route_refresh_interval': route_refresh_interval,
        }
        for layer_idx, layer in enumerate(self.gpt_neox.layers):
            layer.attention = SublinearOceanAttention(
                config, layer_idx=layer_idx, **self.sublinear_kwargs
            )

    def new_cache(self, max_length):
        for layer in self.gpt_neox.layers:
            layer.attention.reset_route()
        return [
            StaticHierarchicalKVCache(
                self.config, max_length=max_length,
                device=DEVICE, dtype=DTYPE,
                **{
                    'block_size': self.sublinear_kwargs['block_size'],
                    'summary_parts': self.sublinear_kwargs['summary_parts'],
                },
            )
            for _ in self.gpt_neox.layers
        ]

    @torch.inference_mode()
    def forward_stream_token(self, input_ids, caches, position):
        hidden_states = self.gpt_neox.embed_in(input_ids.view(1, 1))
        for layer, cache in zip(self.gpt_neox.layers, caches):
            residual = hidden_states
            attention_input = layer.input_layernorm(hidden_states)
            attention_output = layer.attention.forward_token(
                attention_input, cache, position
            )
            if layer.use_parallel_residual:
                mlp_input = layer.post_attention_layernorm(hidden_states)
                mlp_output = layer.mlp(mlp_input)
                hidden_states = residual + attention_output + mlp_output
            else:
                hidden_states = residual + attention_output
                hidden_states = hidden_states + layer.mlp(
                    layer.post_attention_layernorm(hidden_states)
                )
        hidden_states = self.gpt_neox.final_layer_norm(hidden_states)
        return self.embed_out(hidden_states)


def make_sublinear_model(dense_model, max_length=2048, **kwargs):
    sublinear_model = SublinearPythiaForCausalLM(dense_model.config, **kwargs)
    sublinear_model.load_state_dict(dense_model.state_dict(), strict=True)
    sublinear_model = sublinear_model.to(device=DEVICE, dtype=DTYPE).eval()
    return sublinear_model


@torch.inference_mode()
def evaluate_sublinear_perplexity(model, token_ids, max_tokens=2048):
    ids = token_ids[:max_tokens].to(DEVICE)
    if ids.numel() < 2:
        raise ValueError('Для perplexity нужно минимум два токена')
    caches = model.new_cache(ids.numel())
    start = time.perf_counter()
    logits = model.forward_stream_token(ids[:1], caches, 0)
    # Do not call .item() inside this loop: it synchronizes CPU and GPU
    # for every token. Accumulate the loss on the device instead.
    total_nll = torch.zeros((), device=DEVICE, dtype=torch.float32)
    for position in range(1, ids.numel()):
        target = ids[position].view(1)
        total_nll.add_(F.cross_entropy(
            logits[:, -1, :].float(), target, reduction='sum'
        ))
        logits = model.forward_stream_token(ids[position], caches, position)
    synchronize()
    elapsed = time.perf_counter() - start
    mean_nll = (total_nll / (ids.numel() - 1)).item()
    return {
        'mean_nll': mean_nll,
        'perplexity': math.exp(mean_nll),
        'seconds': elapsed,
        'tokens_per_second': (ids.numel() - 1) / elapsed,
        'tokens': ids.numel() - 1,
    }


@torch.inference_mode()
def benchmark_sublinear_generation(model, token_ids, prompt_length=2048, new_tokens=64):
    ids = token_ids[:prompt_length].to(DEVICE)
    caches = model.new_cache(prompt_length + new_tokens)
    synchronize()
    prefill_start = time.perf_counter()
    logits = None
    for position in range(ids.numel()):
        logits = model.forward_stream_token(ids[position], caches, position)
    synchronize()
    prefill_seconds = time.perf_counter() - prefill_start
    next_token = logits[:, -1:, :].argmax(dim=-1)

    synchronize()
    decode_start = time.perf_counter()
    for position in range(prompt_length, prompt_length + new_tokens):
        logits = model.forward_stream_token(next_token[:, 0], caches, position)
        next_token = logits[:, -1:, :].argmax(dim=-1)
    synchronize()
    decode_seconds = time.perf_counter() - decode_start
    return {
        'prefill_seconds': prefill_seconds,
        'prefill_tokens_per_second': prompt_length / prefill_seconds,
        'decode_seconds': decode_seconds,
        'decode_tokens_per_second': new_tokens / decode_seconds,
        'total_seconds': prefill_seconds + decode_seconds,
    }

In [13]:
# Native-context quality benchmark and long-context speed benchmark use
# separate configurations. Pythia quality is only validated up to 2048.
QUALITY_BENCHMARK_CONFIG = {
    'block_size': 64, 'route_blocks': 16, 'beam_width': 32,
    'summary_parts': 4, 'global_blocks': 1, 'local_blocks': 2,
    'route_refresh_interval': 4,
}
LONG_CONTEXT_SPEED_CONFIG = {
    'block_size': 256, 'route_blocks': 16, 'beam_width': 32,
    'summary_parts': 4, 'global_blocks': 1, 'local_blocks': 2,
    'route_refresh_interval': 64,
}

RUN_SUBLINEAR_STREAMING_TEST = True
if RUN_SUBLINEAR_STREAMING_TEST:
    missing = [
        name for name in ('model', 'shakespeare_ids')
        if name not in globals()
    ]
    if missing:
        raise RuntimeError(
            'Не найдены ' + ', '.join(missing) + '. Сначала выполните ячейки '
            'конфигурации, загрузки весов и Tiny Shakespeare (обычно cells 1–14), '
            'затем повторите эту секцию.'
        )
    sublinear_model = make_sublinear_model(
        model, max_length=config.max_position_embeddings,
        **QUALITY_BENCHMARK_CONFIG,
    )
    sublinear_ppl = evaluate_sublinear_perplexity(
        sublinear_model, shakespeare_ids, max_tokens=2048
    )
    print('config:', QUALITY_BENCHMARK_CONFIG)
    print('--- sublinear streaming PPL ---')
    print(sublinear_ppl)


config: {'block_size': 64, 'route_blocks': 16, 'beam_width': 32, 'summary_parts': 4, 'global_blocks': 1, 'local_blocks': 2, 'route_refresh_interval': 4}
--- sublinear streaming PPL ---
{'mean_nll': 3.0724172592163086, 'perplexity': 21.594038024721506, 'seconds': 58.38138393801637, 'tokens_per_second': 35.06254668737048, 'tokens': 2047}


In [14]:
# Speed-only long-context benchmark. PPL is intentionally not computed here:
# contexts above 2048 are outside the trained Pythia-1B context.
LONG_CONTEXT_SPEED_CONFIG = {
    'block_size': 256, 'route_blocks': 16, 'beam_width': 32,
    'summary_parts': 4, 'global_blocks': 1, 'local_blocks': 2,
    'route_refresh_interval': 64,
}
RUN_SUBLINEAR_LONG_CONTEXT_SPEED_TEST = True
if RUN_SUBLINEAR_LONG_CONTEXT_SPEED_TEST:
    missing = [
        name for name in ('model', 'shakespeare_ids')
        if name not in globals()
    ]
    if missing:
        raise RuntimeError(
            'Не найдены ' + ', '.join(missing) + '. Сначала выполните ячейки '
            'конфигурации, загрузки весов и Tiny Shakespeare (обычно cells 1–14), '
            'затем повторите эту секцию.'
        )
    long_model = make_sublinear_model(
        model, max_length=14064, **LONG_CONTEXT_SPEED_CONFIG
    )
    for prompt_length in (2048, 4096, 8192, 14000):
        result = benchmark_sublinear_generation(
            long_model, shakespeare_ids,
            prompt_length=prompt_length, new_tokens=64,
        )
        selected_tokens = min(
            prompt_length,
            LONG_CONTEXT_SPEED_CONFIG['route_blocks']
            * LONG_CONTEXT_SPEED_CONFIG['block_size'],
        )
        print({
            'prompt_length': prompt_length,
            'selected_tokens_max': selected_tokens,
            **result,
        })


{'prompt_length': 2048, 'selected_tokens_max': 2048, 'prefill_seconds': 42.84985773009248, 'prefill_tokens_per_second': 47.79479112626636, 'decode_seconds': 1.4210858789738268, 'decode_tokens_per_second': 45.035983360987814, 'total_seconds': 44.27094360906631}
{'prompt_length': 4096, 'selected_tokens_max': 4096, 'prefill_seconds': 84.97858949401416, 'prefill_tokens_per_second': 48.20037640526523, 'decode_seconds': 1.4523983539547771, 'decode_tokens_per_second': 44.06504580904582, 'total_seconds': 86.43098784796894}
{'prompt_length': 8192, 'selected_tokens_max': 4096, 'prefill_seconds': 174.2461402839981, 'prefill_tokens_per_second': 47.01395386232445, 'decode_seconds': 1.4871460399590433, 'decode_tokens_per_second': 43.03545064193063, 'total_seconds': 175.73328632395715}
{'prompt_length': 14000, 'selected_tokens_max': 4096, 'prefill_seconds': 295.5543910760898, 'prefill_tokens_per_second': 47.368607683435606, 'decode_seconds': 1.3845706668216735, 'decode_tokens_per_second': 46.22371507

## 18. Hybrid sliding-window model: full-scan versus hierarchical routing

This section defines two otherwise identical streaming Pythia models. Both use
the same strict local attention window, global blocks, semantic block budget,
preallocated KV-cache, incremental summaries, and causal exact attention over
the selected candidates. The only routing difference is:

~~~text
HybridFullScanPythia  -> scores every eligible block summary
HybridHierarchicalPythia -> searches the summary tree with beam search
~~~

The local window is always included explicitly as the latest window_size
tokens. It is not counted as a routed block. route_blocks counts global plus
semantic old blocks. The two models must be benchmarked with exactly the same
configuration. The comparison cell is disabled by default because constructing
two 1B-parameter models and long caches can consume substantial GPU memory.

In [27]:
class HybridSlidingWindowAttention(PythiaAttention):
    """Streaming attention with a strict local window and selectable router.

    routing='full_scan' scores all eligible leaf summaries.
    routing='hierarchical' uses the persistent summary tree and beam search.
    """

    def __init__(
        self,
        config,
        layer_idx,
        routing,
        block_size=256,
        window_size=256,
        route_blocks=16,
        beam_width=32,
        summary_parts=4,
        global_blocks=1,
        route_refresh_interval=16,
        collect_route_recall=False,
    ):
        super().__init__(config)
        if routing not in ('full_scan', 'hierarchical'):
            raise ValueError("routing must be 'full_scan' or 'hierarchical'")
        if block_size % summary_parts != 0:
            raise ValueError('block_size must be divisible by summary_parts')
        if window_size <= 0 or route_blocks <= 0:
            raise ValueError('window_size and route_blocks must be positive')

        self.layer_idx = layer_idx
        self.routing = routing
        self.block_size = block_size
        self.window_size = window_size
        self.max_route_blocks = route_blocks
        self.beam_width = beam_width
        self.summary_parts = summary_parts
        self.global_blocks = global_blocks
        self.route_refresh_interval = route_refresh_interval
        self.collect_route_recall = collect_route_recall
        self.reset_route()

    def reset_route(self):
        self._route_cache = None
        self._route_num_blocks = -1
        self._route_age = 0
        self._route_refresh_count = 0
        self._route_recall_sum = 0.0
        self._route_recall_count = 0

    @property
    def mean_route_recall(self):
        if self._route_recall_count == 0:
            return float('nan')
        return self._route_recall_sum / self._route_recall_count

    @torch.no_grad()
    def _update_hierarchy(self, cache, new_key, position):
        block_id = position // cache.block_size
        part_id = (position % cache.block_size) // cache.part_size
        node = cache.leaf_start + block_id
        delta = new_key[0, :, 0, :]
        while True:
            cache.tree_sums[:, node, part_id, :].add_(delta)
            cache.tree_counts[node, part_id].add_(1)
            if node == 0:
                break
            node = (node - 1) // 2

    def _node_scores(self, query_vector, cache, node_ids):
        # query_vector: [heads, head_dim]
        # node_ids: [heads, candidates]
        gather_ids = node_ids[:, :, None, None].expand(
            node_ids.shape[0],
            node_ids.shape[1],
            cache.summary_parts,
            cache.head_dim,
        )
        sums = cache.tree_sums.gather(1, gather_ids)
        counts = cache.tree_counts[node_ids]
        safe_counts = counts.clamp_min(1).to(dtype=sums.dtype)
        summaries = sums / safe_counts.unsqueeze(-1)

        query_norm = F.normalize(query_vector.float(), dim=-1).unsqueeze(1).unsqueeze(2)
        summary_norm = F.normalize(summaries.float(), dim=-1)
        part_scores = (query_norm * summary_norm).sum(dim=-1)
        part_scores = part_scores.masked_fill(counts <= 0, float('-inf'))
        return part_scores.max(dim=-1).values

    def _leaf_scores(self, query_vector, cache, num_blocks):
        block_ids = torch.arange(
            num_blocks, device=query_vector.device, dtype=torch.long
        )
        block_ids = block_ids.view(1, -1).expand(
            self.num_attention_heads, -1
        )
        leaf_nodes = block_ids + cache.leaf_start
        return self._node_scores(query_vector, cache, leaf_nodes)

    def _protected_blocks(self, length, num_blocks, device):
        route_count = min(self.max_route_blocks, num_blocks)
        global_count = min(self.global_blocks, route_count)
        global_ids = torch.arange(
            global_count, device=device, dtype=torch.long
        )

        # Every block touched by the strict local window is protected from
        # semantic selection. This avoids paying for the same block twice.
        local_start = max(0, length - self.window_size)
        local_block_start = local_start // self.block_size
        local_overlap_ids = torch.arange(
            local_block_start, num_blocks, device=device, dtype=torch.long
        )
        protected = torch.unique(torch.cat((global_ids, local_overlap_ids)))
        candidate_mask = torch.ones(
            num_blocks, device=device, dtype=torch.bool
        )
        candidate_mask[protected] = False
        semantic_slots = max(0, route_count - global_count)
        return global_ids, candidate_mask, semantic_slots

    @torch.no_grad()
    def _select_full_scan(self, query, cache, length):
        num_blocks = math.ceil(length / self.block_size)
        global_ids, candidate_mask, semantic_slots = self._protected_blocks(
            length, num_blocks, query.device
        )
        query_vector = query[:, :, 0, :].reshape(
            self.num_attention_heads, self.head_dim
        )
        scores = self._leaf_scores(query_vector, cache, num_blocks)
        scores = scores.masked_fill(~candidate_mask.view(1, -1), float('-inf'))

        available = int(candidate_mask.sum().item())
        take = min(semantic_slots, available)
        if take:
            semantic_ids = torch.topk(
                scores, k=take, dim=-1, largest=True, sorted=True
            ).indices
        else:
            semantic_ids = torch.empty(
                self.num_attention_heads, 0,
                device=query.device,
                dtype=torch.long,
            )

        if global_ids.numel():
            global_part = global_ids.view(1, -1).expand(
                self.num_attention_heads, -1
            )
            route = torch.cat((global_part, semantic_ids), dim=-1)
        else:
            route = semantic_ids
        return route.unsqueeze(0)

    @torch.no_grad()
    def _select_hierarchical(self, query, cache, length):
        num_blocks = math.ceil(length / self.block_size)
        global_ids, candidate_mask, semantic_slots = self._protected_blocks(
            length, num_blocks, query.device
        )
        query_vector = query[:, :, 0, :].reshape(
            self.num_attention_heads, self.head_dim
        )

        beam = min(max(self.beam_width, self.max_route_blocks), num_blocks)
        candidates = torch.zeros(
            self.num_attention_heads, 1,
            device=query.device,
            dtype=torch.long,
        )
        levels = int(math.log2(cache.tree_capacity))

        for _ in range(levels):
            left = candidates * 2 + 1
            right = left + 1
            children = torch.cat((left, right), dim=-1)
            child_scores = self._node_scores(query_vector, cache, children)
            keep = min(beam, children.shape[-1])
            order = torch.topk(
                child_scores, k=keep, dim=-1, largest=True, sorted=True
            ).indices
            candidates = children.gather(-1, order)

        leaf_ids = candidates - cache.leaf_start
        safe_leaf_ids = leaf_ids.clamp(min=0, max=max(0, num_blocks - 1))
        leaf_scores = self._node_scores(
            query_vector,
            cache,
            safe_leaf_ids + cache.leaf_start,
        )
        valid = (
            (leaf_ids >= 0)
            & (leaf_ids < num_blocks)
            & candidate_mask[safe_leaf_ids]
        )
        leaf_scores = leaf_scores.masked_fill(~valid, float('-inf'))

        available_per_head = torch.isfinite(leaf_scores).sum(dim=-1)
        take = min(
            semantic_slots,
            beam,
            int(available_per_head.min().item()),
        )
        if take:
            order = torch.topk(
                leaf_scores, k=take, dim=-1, largest=True, sorted=True
            ).indices
            semantic_ids = safe_leaf_ids.gather(1, order)
        else:
            semantic_ids = torch.empty(
                self.num_attention_heads, 0,
                device=query.device,
                dtype=torch.long,
            )

        if global_ids.numel():
            global_part = global_ids.view(1, -1).expand(
                self.num_attention_heads, -1
            )
            route = torch.cat((global_part, semantic_ids), dim=-1)
        else:
            route = semantic_ids
        return route.unsqueeze(0)

    @torch.no_grad()
    def _record_route_recall(self, exact_route, hierarchical_route):
        exact = exact_route[0]
        approx = hierarchical_route[0]
        head_recalls = []
        for head in range(exact.shape[0]):
            exact_ids = exact[head].unique()
            approx_ids = approx[head].unique()
            if exact_ids.numel() == 0:
                head_recalls.append(1.0)
                continue
            hits = (approx_ids[:, None] == exact_ids[None, :]).any(dim=1)
            head_recalls.append(hits.float().mean().item())
        self._route_recall_sum += sum(head_recalls) / len(head_recalls)
        self._route_recall_count += 1

    @torch.no_grad()
    def _select_route(self, query, cache, length):
        if self.routing == 'full_scan':
            return self._select_full_scan(query, cache, length)

        route = self._select_hierarchical(query, cache, length)
        if self.collect_route_recall:
            exact_route = self._select_full_scan(query, cache, length)
            self._record_route_recall(exact_route, route)
        return route

    def _candidate_tokens(self, route, cache, length):
        local_start = max(0, length - self.window_size)
        local_ids = torch.arange(
            local_start, length, device=route.device, dtype=torch.long
        )
        local_ids = local_ids.view(1, 1, -1).expand(
            1, self.num_attention_heads, -1
        )

        block_offsets = cache.offsets
        block_ids = route[:, :, :, None] * self.block_size + block_offsets
        block_valid = (block_ids < length).reshape(
            1, self.num_attention_heads, -1
        )
        block_ids = block_ids.clamp(max=length - 1).reshape(
            1, self.num_attention_heads, -1
        )

        token_ids = torch.cat((local_ids, block_ids), dim=-1)
        valid = torch.cat((
            torch.ones_like(local_ids, dtype=torch.bool),
            block_valid,
        ), dim=-1)
        gather_ids = token_ids.unsqueeze(-1).expand(
            -1, -1, -1, self.head_dim
        )
        key_ids = gather_ids
        return key_ids, token_ids, valid

    @torch.no_grad()
    def forward_token(self, hidden_states, cache, position):
        batch_size, query_length, _ = hidden_states.shape
        if batch_size != 1 or query_length != 1:
            raise NotImplementedError(
                'Hybrid streaming path supports batch=1 and q_len=1'
            )

        qkv = self.query_key_value(hidden_states)
        qkv = qkv.view(
            1, 1, self.num_attention_heads, 3 * self.head_dim
        ).transpose(1, 2)
        query, new_key, new_value = qkv.chunk(3, dim=-1)

        position_ids = cache.position_ids[position:position + 1]
        cos, sin = self.rotary_emb(position_ids, hidden_states.dtype)
        query, new_key = apply_rotary(
            query, new_key, cos, sin, self.rotary_ndims
        )

        cache.key[:, :, position:position + 1, :].copy_(new_key)
        cache.value[:, :, position:position + 1, :].copy_(new_value)
        self._update_hierarchy(cache, new_key, position)

        length = position + 1
        num_blocks = math.ceil(length / self.block_size)
        must_refresh = (
            self._route_cache is None
            or self._route_age >= self.route_refresh_interval
            or self._route_num_blocks != num_blocks
        )
        if must_refresh:
            route = self._select_route(query, cache, length)
            self._route_cache = route
            self._route_num_blocks = num_blocks
            self._route_age = 0
            self._route_refresh_count += 1
        else:
            route = self._route_cache
            self._route_age += 1

        key_ids, token_ids, valid = self._candidate_tokens(
            route, cache, length
        )
        key_route = cache.key.gather(2, key_ids)
        value_route = cache.value.gather(2, key_ids)
        attention_output = F.scaled_dot_product_attention(
            query,
            key_route,
            value_route,
            attn_mask=valid[:, :, None, :],
            dropout_p=0.0,
            is_causal=False,
        )
        attention_output = attention_output.transpose(1, 2).contiguous()
        attention_output = attention_output.view(1, 1, -1)
        return self.dense(attention_output)


class HybridSlidingWindowPythiaForCausalLM(PythiaForCausalLM):
    def __init__(self, config, routing='hierarchical', **kwargs):
        super().__init__(config)
        self.hybrid_kwargs = dict(kwargs)
        self.hybrid_kwargs['routing'] = routing
        for layer_idx, layer in enumerate(self.gpt_neox.layers):
            layer.attention = HybridSlidingWindowAttention(
                config,
                layer_idx=layer_idx,
                routing=routing,
                **kwargs,
            )

    def new_cache(self, max_length):
        for layer in self.gpt_neox.layers:
            layer.attention.reset_route()
        parameter = next(self.parameters())
        return [
            StaticHierarchicalKVCache(
                self.config,
                max_length=max_length,
                device=parameter.device,
                dtype=parameter.dtype,
                block_size=self.hybrid_kwargs['block_size'],
                summary_parts=self.hybrid_kwargs['summary_parts'],
            )
            for _ in self.gpt_neox.layers
        ]

    @torch.inference_mode()
    def forward_stream_token(self, input_ids, caches, position):
        hidden_states = self.gpt_neox.embed_in(input_ids.view(1, 1))
        for layer, cache in zip(self.gpt_neox.layers, caches):
            residual = hidden_states
            attention_input = layer.input_layernorm(hidden_states)
            attention_output = layer.attention.forward_token(
                attention_input, cache, position
            )
            if layer.use_parallel_residual:
                mlp_input = layer.post_attention_layernorm(hidden_states)
                mlp_output = layer.mlp(mlp_input)
                hidden_states = residual + attention_output + mlp_output
            else:
                hidden_states = residual + attention_output
                hidden_states = hidden_states + layer.mlp(
                    layer.post_attention_layernorm(hidden_states)
                )
        hidden_states = self.gpt_neox.final_layer_norm(hidden_states)
        return self.embed_out(hidden_states)

    def reset_hybrid_routes(self):
        for layer in self.gpt_neox.layers:
            layer.attention.reset_route()


def make_hybrid_sliding_model(dense_model, routing='hierarchical', **kwargs):
    routed_model = HybridSlidingWindowPythiaForCausalLM(
        dense_model.config,
        routing=routing,
        **kwargs,
    )
    routed_model.load_state_dict(dense_model.state_dict(), strict=True)
    return routed_model.to(
        device=next(dense_model.parameters()).device,
        dtype=next(dense_model.parameters()).dtype,
    ).eval()


def make_hybrid_routing_pair(dense_model, **kwargs):
    return {
        'full_scan': make_hybrid_sliding_model(
            dense_model, routing='full_scan', **kwargs
        ),
        'hierarchical': make_hybrid_sliding_model(
            dense_model, routing='hierarchical', **kwargs
        ),
    }


@torch.inference_mode()
def evaluate_hybrid_perplexity(model, token_ids, max_tokens=2048):
    model.reset_hybrid_routes()
    ids = token_ids[:max_tokens].to(next(model.parameters()).device)
    if ids.numel() < 2:
        raise ValueError('perplexity requires at least two tokens')
    caches = model.new_cache(ids.numel())
    total_nll = torch.zeros(
        (), device=ids.device, dtype=torch.float32
    )
    start = time.perf_counter()
    logits = model.forward_stream_token(ids[:1], caches, 0)
    for position in range(1, ids.numel()):
        target = ids[position].view(1)
        total_nll.add_(F.cross_entropy(
            logits[:, -1, :].float(),
            target,
            reduction='sum',
        ))
        logits = model.forward_stream_token(
            ids[position], caches, position
        )
    synchronize()
    elapsed = time.perf_counter() - start
    refreshes = sum(
        layer.attention._route_refresh_count
        for layer in model.gpt_neox.layers
    )
    recalls = [
        layer.attention.mean_route_recall
        for layer in model.gpt_neox.layers
        if layer.attention._route_recall_count
    ]
    mean_nll = (total_nll / (ids.numel() - 1)).item()
    return {
        'mean_nll': mean_nll,
        'perplexity': math.exp(mean_nll),
        'seconds': elapsed,
        'tokens_per_second': (ids.numel() - 1) / elapsed,
        'tokens': ids.numel() - 1,
        'route_refreshes_all_layers': refreshes,
        'mean_hierarchical_route_recall': (
            sum(recalls) / len(recalls) if recalls else None
        ),
    }


@torch.inference_mode()
def benchmark_hybrid_generation(
    model,
    token_ids,
    prompt_length=2048,
    new_tokens=64,
):
    ids = token_ids[:prompt_length].to(next(model.parameters()).device)
    caches = model.new_cache(prompt_length + new_tokens)
    synchronize()
    prefill_start = time.perf_counter()
    logits = None
    for position in range(ids.numel()):
        logits = model.forward_stream_token(
            ids[position], caches, position
        )
    synchronize()
    prefill_seconds = time.perf_counter() - prefill_start
    next_token = logits[:, -1:, :].argmax(dim=-1)

    synchronize()
    decode_start = time.perf_counter()
    for position in range(prompt_length, prompt_length + new_tokens):
        logits = model.forward_stream_token(
            next_token[:, 0], caches, position
        )
        next_token = logits[:, -1:, :].argmax(dim=-1)
    synchronize()
    decode_seconds = time.perf_counter() - decode_start
    return {
        'prompt_length': prompt_length,
        'prefill_seconds': prefill_seconds,
        'prefill_tokens_per_second': prompt_length / prefill_seconds,
        'decode_seconds': decode_seconds,
        'decode_tokens_per_second': new_tokens / decode_seconds,
        'total_seconds': prefill_seconds + decode_seconds,
        'total_tokens_per_second': (
            (prompt_length + new_tokens)
            / (prefill_seconds + decode_seconds)
        ),
    }


def hybrid_route_stats(model):
    layers = [
        layer.attention for layer in model.gpt_neox.layers
    ]
    return {
        'routing': layers[0].routing,
        'route_refreshes_all_layers': sum(
            layer._route_refresh_count for layer in layers
        ),
        'mean_hierarchical_route_recall': (
            sum(
                layer.mean_route_recall for layer in layers
                if layer._route_recall_count
            )
            / sum(1 for layer in layers if layer._route_recall_count)
            if any(layer._route_recall_count for layer in layers)
            else None
        ),
    }


def run_hybrid_routing_comparison(
    dense_model,
    token_ids,
    quality_config,
    quality_tokens=2048,
    speed_prompt_lengths=(2048, 4096, 8192, 14000),
    speed_new_tokens=64,
    collect_hierarchical_recall=False,
):
    models = make_hybrid_routing_pair(dense_model, **quality_config)
    if collect_hierarchical_recall:
        for layer in models['hierarchical'].gpt_neox.layers:
            layer.attention.collect_route_recall = True

    print('configuration:', quality_config)
    results = {}
    for name, routed_model in models.items():
        result = evaluate_hybrid_perplexity(
            routed_model, token_ids, max_tokens=quality_tokens
        )
        results[name] = {
            'quality': result,
            'quality_route_stats': hybrid_route_stats(routed_model),
        }
        print('---', name, 'quality ---')
        print(result)

    # Route recall requires an extra exact full-scan route and must not
    # remain enabled during the speed benchmark.
    for layer in models['hierarchical'].gpt_neox.layers:
        layer.attention.collect_route_recall = False

    for prompt_length in speed_prompt_lengths:
        if token_ids.numel() < prompt_length:
            continue
        for name, routed_model in models.items():
            result = benchmark_hybrid_generation(
                routed_model,
                token_ids,
                prompt_length=prompt_length,
                new_tokens=speed_new_tokens,
            )
            results[name, prompt_length] = result
            print('---', name, 'speed ---')
            print(result)

    for name, routed_model in models.items():
        results[name, 'speed_route_stats'] = hybrid_route_stats(routed_model)
    return models, results


HYBRID_QUALITY_CONFIG = {
    'block_size': 64,
    'window_size': 256,
    'route_blocks': 16,
    'beam_width': 32,
    'summary_parts': 4,
    'global_blocks': 1,
    'route_refresh_interval': 4,
}
HYBRID_LONG_CONTEXT_CONFIG = {
    'block_size': 256,
    'window_size': 256,
    'route_blocks': 16,
    'beam_width': 32,
    'summary_parts': 4,
    'global_blocks': 1,
    'route_refresh_interval': 64,
}

In [ ]:
# Disabled by default to avoid allocating two additional 1B models
# unintentionally. Set True after the base model and shakespeare_ids exist.
RUN_HYBRID_ROUTING_COMPARISON = True
if RUN_HYBRID_ROUTING_COMPARISON:
    missing = [
        name for name in ('model', 'shakespeare_ids')
        if name not in globals()
    ]
    if missing:
        raise RuntimeError(
            'Missing ' + ', '.join(missing) + '. Execute the Pythia '
            'weights and Shakespeare cells first.'
        )

    # If memory is tight, delete previous Ocean/sublinear instances first:
    # del ocean_model, sublinear_model, long_model
    # clear_gpu_cache()
    hybrid_models, hybrid_results = run_hybrid_routing_comparison(
        model,
        shakespeare_ids,
        quality_config=HYBRID_QUALITY_CONFIG,
        quality_tokens=2048,
        speed_prompt_lengths=(2048,),
        collect_hierarchical_recall=True,
    )

configuration: {'block\_size': 64, 'window\_size': 256, 'route\_blocks': 16, 'beam\_width': 32, 'summary\_parts': 4, 'global\_blocks': 1, 'route\_refresh\_interval': 4}


\--- full\_scan quality ---

{'mean\_nll': 3.0785298347473145, 'perplexity': 21.726437451672993, 'seconds': 49.3280175679829, 'tokens\_per\_second': 41.49771470501253, 'tokens': 2047, 'route\_refreshes\_all\_layers': 6656, 'mean\_hierarchical\_route\_recall': None}

\--- hierarchical quality ---

{'mean\_nll': 3.0785298347473145, 'perplexity': 21.726437451672993, 'seconds': 64.2115542460233, 'tokens\_per\_second': 31.878997853828984, 'tokens': 2047, 'route\_refreshes\_all\_layers': 6656, 'mean\_hierarchical\_route\_recall': 1.0}

\--- full\_scan speed ---

{'prompt\_length': 2048, 'prefill\_seconds': 39.912438585888594, 'prefill\_tokens\_per\_second': 51.31232449234733, 'decode\_seconds': 1.2558592718560249, 'decode\_tokens\_per\_second': 50.96112393661345, 'total\_seconds': 41.16829785774462, 'total\_tokens\_per\_second': 51.301610945828514}

\--- hierarchical speed ---

{'prompt\_length': 2048, 'prefill\_seconds': 54.40640943311155, 'prefill\_tokens\_per\_second': 37.64262375222274, 'decode\_seconds': 1.6995897251181304, 'decode\_tokens\_per\_second': 37.656146688902616, 'total\_seconds': 56.10599915822968, 'total\_tokens\_per\_second': 37.643033395479776}

## 19. Independent 14K speed benchmarks

The following cells benchmark the two routing implementations separately. The
14K prompt is outside the validated Pythia-1B training context, so these cells
measure speed only. Each cell creates one routed model, runs the same 14K
prefill plus 64-token decode benchmark, deletes the model, and clears the GPU
cache before returning control.

In [28]:
def clear_gpu_cache():
    """Release Python references and cached CUDA allocator blocks."""
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


HYBRID_14K_SPEED_CONFIG = {
    'block_size': 256,
    'window_size': 256,
    'route_blocks': 16,
    'beam_width': 32,
    'summary_parts': 4,
    'global_blocks': 1,
    'route_refresh_interval': 64,
}
HYBRID_14K_PROMPT_LENGTH = 14_000
HYBRID_14K_NEW_TOKENS = 64

### 19.1 Full-scan routing: 14K speed benchmark

This cell contains no hierarchical route search. It scans all eligible block
summaries at every route refresh.

In [18]:
RUN_HYBRID_FULL_SCAN_14K_BENCHMARK = True

if RUN_HYBRID_FULL_SCAN_14K_BENCHMARK:
    missing = [
        name for name in ('model', 'shakespeare_ids')
        if name not in globals()
    ]
    if missing:
        raise RuntimeError(
            'Missing ' + ', '.join(missing) + '. Execute the Pythia weights '
            'and Shakespeare cells first.'
        )

    clear_gpu_cache()
    full_scan_14k_model = make_hybrid_sliding_model(
        model,
        routing='full_scan',
        **HYBRID_14K_SPEED_CONFIG,
    )
    try:
        full_scan_14k_result = benchmark_hybrid_generation(
            full_scan_14k_model,
            shakespeare_ids,
            prompt_length=HYBRID_14K_PROMPT_LENGTH,
            new_tokens=HYBRID_14K_NEW_TOKENS,
        )
        full_scan_14k_result['routing'] = 'full_scan'
        full_scan_14k_result['config'] = HYBRID_14K_SPEED_CONFIG
        full_scan_14k_result['selected_tokens_upper_bound'] = (
            HYBRID_14K_SPEED_CONFIG['window_size']
            + HYBRID_14K_SPEED_CONFIG['route_blocks']
            * HYBRID_14K_SPEED_CONFIG['block_size']
        )
        print('--- hybrid full-scan 14K speed ---')
        print(full_scan_14k_result)
    finally:
        del full_scan_14k_model
        clear_gpu_cache()

--- hybrid full-scan 14K speed ---
{'prompt_length': 14000, 'prefill_seconds': 256.2907672780566, 'prefill_tokens_per_second': 54.625455878443844, 'decode_seconds': 1.1768343839794397, 'decode_tokens_per_second': 54.38318328496266, 'total_seconds': 257.46760166203603, 'total_tokens_per_second': 54.62434849749003, 'routing': 'full_scan', 'config': {'block_size': 256, 'window_size': 256, 'route_blocks': 16, 'beam_width': 32, 'summary_parts': 4, 'global_blocks': 1, 'route_refresh_interval': 64}, 'selected_tokens_upper_bound': 4352}


--- hybrid full-scan 14K speed ---

{'prompt_length': 14000, 'prefill_seconds': 256.2907672780566, 'prefill_tokens_per_second': 54.625455878443844, 'decode_seconds': 1.1768343839794397, 'decode_tokens_per_second': 54.38318328496266, 'total_seconds': 257.46760166203603, 'total_tokens_per_second': 54.62434849749003, 'routing': 'full_scan', 'config': {'block_size': 256, 'window_size': 256, 'route_blocks': 16, 'beam_width': 32, 'summary_parts': 4, 'global_blocks': 1, 'route_refresh_interval': 64}, 'selected_tokens_upper_bound': 4352}

### 19.2 Hierarchical routing: 14K speed benchmark

This cell uses the same model configuration and benchmark protocol, but
replaces full summary scanning with hierarchical beam search.

In [19]:
RUN_HYBRID_HIERARCHICAL_14K_BENCHMARK = True

if RUN_HYBRID_HIERARCHICAL_14K_BENCHMARK:
    missing = [
        name for name in ('model', 'shakespeare_ids')
        if name not in globals()
    ]
    if missing:
        raise RuntimeError(
            'Missing ' + ', '.join(missing) + '. Execute the Pythia weights '
            'and Shakespeare cells first.'
        )

    clear_gpu_cache()
    hierarchical_14k_model = make_hybrid_sliding_model(
        model,
        routing='hierarchical',
        **HYBRID_14K_SPEED_CONFIG,
    )
    try:
        hierarchical_14k_result = benchmark_hybrid_generation(
            hierarchical_14k_model,
            shakespeare_ids,
            prompt_length=HYBRID_14K_PROMPT_LENGTH,
            new_tokens=HYBRID_14K_NEW_TOKENS,
        )
        hierarchical_14k_result['routing'] = 'hierarchical'
        hierarchical_14k_result['config'] = HYBRID_14K_SPEED_CONFIG
        hierarchical_14k_result['selected_tokens_upper_bound'] = (
            HYBRID_14K_SPEED_CONFIG['window_size']
            + HYBRID_14K_SPEED_CONFIG['route_blocks']
            * HYBRID_14K_SPEED_CONFIG['block_size']
        )
        print('--- hybrid hierarchical 14K speed ---')
        print(hierarchical_14k_result)
    finally:
        del hierarchical_14k_model
        clear_gpu_cache()

--- hybrid hierarchical 14K speed ---
{'prompt_length': 14000, 'prefill_seconds': 266.84927875804715, 'prefill_tokens_per_second': 52.46407284725634, 'decode_seconds': 1.1980524610262364, 'decode_tokens_per_second': 53.420031327491635, 'total_seconds': 268.0473312190734, 'total_tokens_per_second': 52.468345556873246, 'routing': 'hierarchical', 'config': {'block_size': 256, 'window_size': 256, 'route_blocks': 16, 'beam_width': 32, 'summary_parts': 4, 'global_blocks': 1, 'route_refresh_interval': 64}, 'selected_tokens_upper_bound': 4352}


--- hybrid hierarchical 14K speed ---

{'prompt_length': 14000, 'prefill_seconds': 266.84927875804715, 'prefill_tokens_per_second': 52.46407284725634, 'decode_seconds': 1.1980524610262364, 'decode_tokens_per_second': 53.420031327491635, 'total_seconds': 268.0473312190734, 'total_tokens_per_second': 52.468345556873246, 'routing': 'hierarchical', 'config': {'block_size': 256, 'window_size': 256, 'route_blocks': 16, 'beam_width': 32, 'summary_parts': 4, 'global_blocks': 1, 'route_refresh_interval': 64}, 'selected_tokens_upper_bound': 4352}

Для контекста 14K в данной реализации full-scan лучше: он быстрее примерно на 4%, а иерархический routing пока не даёт измеримого преимущества

### 19.3 Dense baseline: 14K speed benchmark

This is the dense Pythia-1B baseline with the same 14,000-token prompt and
64-token greedy decode. The result is speed-only because 14K exceeds the
validated Pythia-1B training context.

In [21]:
RUN_DENSE_14K_BENCHMARK = True

if RUN_DENSE_14K_BENCHMARK:
    missing = [
        name for name in ('model', 'shakespeare_ids')
        if name not in globals()
    ]
    if missing:
        raise RuntimeError(
            'Missing ' + ', '.join(missing) + '. Execute the Pythia weights '
            'and Shakespeare cells first.'
        )

    clear_gpu_cache()
    try:
        dense_14k_result = benchmark_generation(
            model,
            shakespeare_ids,
            prompt_length=HYBRID_14K_PROMPT_LENGTH,
            new_tokens=HYBRID_14K_NEW_TOKENS,
            allow_untrained_context=True,
        )
        # Do not print the generated 14K text in the benchmark report.
        dense_14k_result.pop('output', None)
        dense_14k_result['routing'] = 'dense'
        dense_14k_result['prompt_length'] = HYBRID_14K_PROMPT_LENGTH
        dense_14k_result['new_tokens'] = HYBRID_14K_NEW_TOKENS
        print('--- dense 14K speed ---')
        print(dense_14k_result)
    finally:
        clear_gpu_cache()

--- dense 14K speed ---
{'prefill_seconds': 6.855087988078594, 'prefill_tokens_per_second': 2042.2786730596067, 'decode_seconds': 1.6783228998538107, 'decode_tokens_per_second': 37.53747267911769, 'total_seconds': 8.533410887932405, 'total_tokens_per_second': 7.499931837397651, 'routing': 'dense', 'prompt_length': 14000, 'new_tokens': 64}


--- dense 14K speed ---

{'prefill_seconds': 6.855087988078594, 'prefill_tokens_per_second': 2042.2786730596067, 'decode_seconds': 1.6783228998538107, 'decode_tokens_per_second': 37.53747267911769, 'total_seconds': 8.533410887932405, 'total_tokens_per_second': 7.499931837397651, 'routing': 'dense', 'prompt_length': 14000, 'new_tokens': 64}

## 20. Fast chunked prefill for hybrid routing

The previous hybrid benchmark filled the custom KV-cache with one
token-by-token call. That is useful for a simple reference implementation but
is not a fair end-to-end prefill implementation. The following model processes
the prompt in chunks:

~~~text
chunk QKV projection -> append chunk K/V and summaries
                    -> one route per chunk
                    -> batched causal attention for all queries in the chunk
                    -> MLP for the whole chunk
~~~

The local-window and routed-block candidates are identical for all queries in a
chunk, while the causal mask prevents a query from seeing future tokens. This
is an optimized approximation: it does not reproduce the old per-token route
exactly, so perplexity must be measured separately.

In [29]:
class FastHybridSlidingWindowAttention(HybridSlidingWindowAttention):
    """Hybrid attention with vectorized chunked prefill."""

    @torch.no_grad()
    def _append_prefill_chunk(self, cache, new_key, new_value, start):
        chunk_length = new_key.shape[2]
        end = start + chunk_length
        cache.key[:, :, start:end, :].copy_(new_key)
        cache.value[:, :, start:end, :].copy_(new_value)

        positions = torch.arange(
            start, end, device=new_key.device, dtype=torch.long
        )
        block_ids = positions // cache.block_size
        part_ids = (
            (positions % cache.block_size) // cache.part_size
        )
        flat_ids = block_ids * cache.summary_parts + part_ids
        max_blocks = cache.tree_capacity

        # Update all token summaries in one index_add per attention head.
        for head in range(self.num_attention_heads):
            leaf_sums = cache.tree_sums[
                head,
                cache.leaf_start:cache.leaf_start + max_blocks,
            ].reshape(-1, cache.head_dim)
            leaf_sums.index_add_(
                0, flat_ids, new_key[0, head]
            )

        leaf_counts = cache.tree_counts[
            cache.leaf_start:cache.leaf_start + max_blocks
        ].reshape(-1)
        leaf_counts.index_add_(
            0,
            flat_ids,
            torch.ones_like(flat_ids, dtype=leaf_counts.dtype),
        )

        # Recompute only ancestors of blocks touched by this chunk.
        for block_id in torch.unique(block_ids).tolist():
            node = cache.leaf_start + int(block_id)
            while node > 0:
                node = (node - 1) // 2
                left = 2 * node + 1
                right = left + 1
                cache.tree_sums[:, node].copy_(
                    cache.tree_sums[:, left]
                    + cache.tree_sums[:, right]
                )
                cache.tree_counts[node].copy_(
                    cache.tree_counts[left]
                    + cache.tree_counts[right]
                )

    def _candidate_tokens_prefill(self, route, cache, start, end):
        # Union of every local-window token needed by any query in the
        # chunk. Causal masking removes tokens future to each query.
        local_start = max(0, start - self.window_size + 1)
        local_ids = torch.arange(
            local_start, end, device=route.device, dtype=torch.long
        )
        local_ids = local_ids.view(1, 1, -1).expand(
            1, self.num_attention_heads, -1
        )
        block_offsets = cache.offsets
        block_ids = route[:, :, :, None] * self.block_size + block_offsets
        block_valid = (block_ids < end).reshape(
            1, self.num_attention_heads, -1
        )
        block_ids = block_ids.clamp(max=end - 1).reshape(
            1, self.num_attention_heads, -1
        )
        token_ids = torch.cat((local_ids, block_ids), dim=-1)
        valid = torch.cat((
            torch.ones_like(local_ids, dtype=torch.bool),
            block_valid,
        ), dim=-1)
        key_ids = token_ids.unsqueeze(-1).expand(
            -1, -1, -1, self.head_dim
        )
        return key_ids, token_ids, valid

    @torch.no_grad()
    def forward_prefill_chunk(self, hidden_states, cache, start):
        batch_size, query_length, _ = hidden_states.shape
        if batch_size != 1:
            raise NotImplementedError(
                'Fast hybrid prefill supports batch=1'
            )

        qkv = self.query_key_value(hidden_states)
        qkv = qkv.view(
            1,
            query_length,
            self.num_attention_heads,
            3 * self.head_dim,
        ).transpose(1, 2)
        query, new_key, new_value = qkv.chunk(3, dim=-1)

        end = start + query_length
        position_ids = cache.position_ids[start:end]
        cos, sin = self.rotary_emb(position_ids, hidden_states.dtype)
        query, new_key = apply_rotary(
            query, new_key, cos, sin, self.rotary_ndims
        )

        if start == 0:
            route = torch.empty(
                1,
                self.num_attention_heads,
                0,
                device=hidden_states.device,
                dtype=torch.long,
            )
        else:
            # Use the final query in the chunk as the routing signal.
            route = self._select_route(
                query[:, :, -1:, :], cache, start
            )

        self._append_prefill_chunk(
            cache, new_key, new_value, start
        )

        # The route was built for the prefix before this chunk. The local
        # candidate union below contains the current chunk and is causal-masked.
        self._route_cache = route
        self._route_num_blocks = math.ceil(end / self.block_size)
        self._route_age = 0
        self._route_refresh_count += 1

        key_ids, token_ids, valid = self._candidate_tokens_prefill(
            route, cache, start, end
        )
        key_route = cache.key.gather(2, key_ids)
        value_route = cache.value.gather(2, key_ids)

        query_positions = torch.arange(
            start, end, device=hidden_states.device, dtype=torch.long
        )
        causal = token_ids[:, :, None, :] <= query_positions.view(
            1, 1, query_length, 1
        )
        attention_mask = valid[:, :, None, :] & causal

        attention_output = F.scaled_dot_product_attention(
            query,
            key_route,
            value_route,
            attn_mask=attention_mask,
            dropout_p=0.0,
            is_causal=False,
        )
        attention_output = attention_output.transpose(1, 2).contiguous()
        attention_output = attention_output.view(
            1, query_length, -1
        ).contiguous()
        return self.dense(attention_output)


class FastHybridSlidingWindowPythiaForCausalLM(
    HybridSlidingWindowPythiaForCausalLM
):
    def __init__(self, config, routing='hierarchical', **kwargs):
        super().__init__(config, routing=routing, **kwargs)
        attention_kwargs = dict(self.hybrid_kwargs)
        attention_kwargs.pop('routing', None)
        for layer_idx, layer in enumerate(self.gpt_neox.layers):
            layer.attention = FastHybridSlidingWindowAttention(
                config,
                layer_idx=layer_idx,
                routing=routing,
                **attention_kwargs,
            )

    @torch.inference_mode()
    def forward_prefill(self, input_ids, chunk_size=64, max_length=None):
        ids = input_ids.view(1, -1)
        prompt_length = ids.shape[1]
        if max_length is None:
            max_length = prompt_length

        caches = self.new_cache(max_length)
        logits = None
        for start in range(0, prompt_length, chunk_size):
            end = min(start + chunk_size, prompt_length)
            hidden_states = self.gpt_neox.embed_in(
                ids[:, start:end]
            )

            for layer, cache in zip(self.gpt_neox.layers, caches):
                residual = hidden_states
                attention_input = layer.input_layernorm(hidden_states)
                attention_output = layer.attention.forward_prefill_chunk(
                    attention_input,
                    cache,
                    start,
                )

                if layer.use_parallel_residual:
                    mlp_input = layer.post_attention_layernorm(
                        hidden_states
                    )
                    mlp_output = layer.mlp(mlp_input)
                    hidden_states = (
                        residual + attention_output + mlp_output
                    )
                else:
                    hidden_states = residual + attention_output
                    hidden_states = hidden_states + layer.mlp(
                        layer.post_attention_layernorm(hidden_states)
                    )

            hidden_states = self.gpt_neox.final_layer_norm(hidden_states)
            logits = self.embed_out(hidden_states)

        return logits, caches


def make_fast_hybrid_sliding_model(
    dense_model, routing='hierarchical', **kwargs
):
    routed_model = FastHybridSlidingWindowPythiaForCausalLM(
        dense_model.config,
        routing=routing,
        **kwargs,
    )
    routed_model.load_state_dict(dense_model.state_dict(), strict=True)
    return routed_model.to(
        device=next(dense_model.parameters()).device,
        dtype=next(dense_model.parameters()).dtype,
    ).eval()


@torch.inference_mode()
def benchmark_fast_hybrid_generation(
    model,
    token_ids,
    prompt_length=14_000,
    new_tokens=64,
    chunk_size=64,
):
    device = next(model.parameters()).device
    ids = token_ids[:prompt_length].to(device)
    synchronize()

    prefill_start = time.perf_counter()
    logits, caches = model.forward_prefill(
        ids,
        chunk_size=chunk_size,
        max_length=prompt_length + new_tokens,
    )
    synchronize()
    prefill_seconds = time.perf_counter() - prefill_start

    next_token = logits[:, -1:, :].argmax(dim=-1)
    synchronize()
    decode_start = time.perf_counter()
    for position in range(prompt_length, prompt_length + new_tokens):
        logits = model.forward_stream_token(
            next_token[:, 0],
            caches,
            position,
        )
        next_token = logits[:, -1:, :].argmax(dim=-1)
    synchronize()
    decode_seconds = time.perf_counter() - decode_start

    return {
        'prompt_length': prompt_length,
        'chunk_size': chunk_size,
        'prefill_seconds': prefill_seconds,
        'prefill_tokens_per_second': (
            prompt_length / prefill_seconds
        ),
        'decode_seconds': decode_seconds,
        'decode_tokens_per_second': (
            new_tokens / decode_seconds
        ),
        'total_seconds': prefill_seconds + decode_seconds,
    }


FAST_HYBRID_14K_CONFIG = {
    'block_size': 256,
    'window_size': 256,
    'route_blocks': 16,
    'beam_width': 32,
    'summary_parts': 4,
    'global_blocks': 1,
    'route_refresh_interval': 64,
}
FAST_HYBRID_PREFILL_CHUNK_SIZE = 64

### 20.1 Fast full-scan hybrid: 14K speed benchmark

This benchmark measures the chunked prefill implementation with full-scan
routing. It is speed-only at 14K.

In [23]:
RUN_FAST_FULL_SCAN_14K_BENCHMARK = True

if RUN_FAST_FULL_SCAN_14K_BENCHMARK:
    missing = [
        name for name in ('model', 'shakespeare_ids')
        if name not in globals()
    ]
    if missing:
        raise RuntimeError(
            'Missing ' + ', '.join(missing) + '. Execute the Pythia '
            'weights and Shakespeare cells first.'
        )

    clear_gpu_cache()
    fast_full_scan_model = make_fast_hybrid_sliding_model(
        model,
        routing='full_scan',
        **FAST_HYBRID_14K_CONFIG,
    )
    try:
        fast_full_scan_result = benchmark_fast_hybrid_generation(
            fast_full_scan_model,
            shakespeare_ids,
            prompt_length=14_000,
            new_tokens=64,
            chunk_size=FAST_HYBRID_PREFILL_CHUNK_SIZE,
        )
        fast_full_scan_result['routing'] = 'full_scan'
        print('--- fast hybrid full-scan 14K ---')
        print(fast_full_scan_result)
    finally:
        del fast_full_scan_model
        clear_gpu_cache()

--- fast hybrid full-scan 14K ---
{'prompt_length': 14000, 'chunk_size': 64, 'prefill_seconds': 16.932695048861206, 'prefill_tokens_per_second': 826.8028190197377, 'decode_seconds': 1.1759090321138501, 'decode_tokens_per_second': 54.42597875530528, 'total_seconds': 18.108604080975056, 'routing': 'full_scan'}


--- fast hybrid full-scan 14K ---

{'prompt_length': 14000, 'chunk_size': 64, 'prefill_seconds': 16.932695048861206, 'prefill_tokens_per_second': 826.8028190197377, 'decode_seconds': 1.1759090321138501, 'decode_tokens_per_second': 54.42597875530528, 'total_seconds': 18.108604080975056, 'routing': 'full_scan'}

### 20.2 Fast hierarchical hybrid: 14K speed benchmark

This benchmark uses the same chunked prefill path and configuration, changing
only the routing policy.

In [24]:
RUN_FAST_HIERARCHICAL_14K_BENCHMARK = True

if RUN_FAST_HIERARCHICAL_14K_BENCHMARK:
    missing = [
        name for name in ('model', 'shakespeare_ids')
        if name not in globals()
    ]
    if missing:
        raise RuntimeError(
            'Missing ' + ', '.join(missing) + '. Execute the Pythia '
            'weights and Shakespeare cells first.'
        )

    clear_gpu_cache()
    fast_hierarchical_model = make_fast_hybrid_sliding_model(
        model,
        routing='hierarchical',
        **FAST_HYBRID_14K_CONFIG,
    )
    try:
        fast_hierarchical_result = benchmark_fast_hybrid_generation(
            fast_hierarchical_model,
            shakespeare_ids,
            prompt_length=14_000,
            new_tokens=64,
            chunk_size=FAST_HYBRID_PREFILL_CHUNK_SIZE,
        )
        fast_hierarchical_result['routing'] = 'hierarchical'
        print('--- fast hybrid hierarchical 14K ---')
        print(fast_hierarchical_result)
    finally:
        del fast_hierarchical_model
        clear_gpu_cache()

--- fast hybrid hierarchical 14K ---
{'prompt_length': 14000, 'chunk_size': 64, 'prefill_seconds': 24.907852266915143, 'prefill_tokens_per_second': 562.0717454871075, 'decode_seconds': 1.1743274361360818, 'decode_tokens_per_second': 54.499280209769054, 'total_seconds': 26.082179703051224, 'routing': 'hierarchical'}


--- fast hybrid hierarchical 14K ---

{'prompt_length': 14000, 'chunk_size': 64, 'prefill_seconds': 24.907852266915143, 'prefill_tokens_per_second': 562.0717454871075, 'decode_seconds': 1.1743274361360818, 'decode_tokens_per_second': 54.499280209769054, 'total_seconds': 26.082179703051224, 'routing': 'hierarchical'}

Почему hierarchical снова проигрывает full-scan:
- 14K содержит около 55 блоков;
- full-scan делает один векторизованный score по summaries;
- hierarchical выполняет несколько уровней дерева, topk, gather и дополнительные GPU kernel launches;
- для 55 блоков и beam_width=32 асимптотическое преимущество ещё недостаточно велико.

## 21. Perplexity validation of chunked prefill

Chunked prefill changes the routing granularity, so its speedup cannot be
accepted without a quality check. The following evaluator accounts for the
prediction crossing a chunk boundary and reports perplexity for the complete
2048-token native-context fragment.

In [30]:
@torch.inference_mode()
def _fast_forward_prefill_with_loss(
    self,
    input_ids,
    chunk_size=64,
    max_length=None,
):
    ids = input_ids.view(1, -1)
    prompt_length = ids.shape[1]
    if max_length is None:
        max_length = prompt_length

    caches = self.new_cache(max_length)
    total_nll = torch.zeros(
        (), device=ids.device, dtype=torch.float32
    )
    token_count = 0
    previous_last_logits = None

    for start in range(0, prompt_length, chunk_size):
        end = min(start + chunk_size, prompt_length)
        hidden_states = self.gpt_neox.embed_in(ids[:, start:end])

        for layer, cache in zip(self.gpt_neox.layers, caches):
            residual = hidden_states
            attention_input = layer.input_layernorm(hidden_states)
            attention_output = layer.attention.forward_prefill_chunk(
                attention_input,
                cache,
                start,
            )

            if layer.use_parallel_residual:
                mlp_input = layer.post_attention_layernorm(
                    hidden_states
                )
                mlp_output = layer.mlp(mlp_input)
                hidden_states = (
                    residual + attention_output + mlp_output
                )
            else:
                hidden_states = residual + attention_output
                hidden_states = hidden_states + layer.mlp(
                    layer.post_attention_layernorm(hidden_states)
                )

        hidden_states = self.gpt_neox.final_layer_norm(hidden_states)
        logits = self.embed_out(hidden_states)

        # The previous chunk's final logit predicts the first token of
        # this chunk. The remaining logits predict within-chunk targets.
        if previous_last_logits is not None:
            total_nll.add_(F.cross_entropy(
                previous_last_logits[:, -1, :].float(),
                ids[:, start],
                reduction='sum',
            ))
            token_count += 1

        if end - start > 1:
            chunk_logits = logits[:, :-1, :].reshape(-1, logits.shape[-1])
            chunk_targets = ids[:, start + 1:end].reshape(-1)
            total_nll.add_(F.cross_entropy(
                chunk_logits.float(),
                chunk_targets,
                reduction='sum',
            ))
            token_count += end - start - 1

        previous_last_logits = logits[:, -1:, :]

    return total_nll, token_count


FastHybridSlidingWindowPythiaForCausalLM.forward_prefill_loss = (
    _fast_forward_prefill_with_loss
)


@torch.inference_mode()
def evaluate_fast_hybrid_perplexity(
    model,
    token_ids,
    max_tokens=2048,
    chunk_size=64,
):
    ids = token_ids[:max_tokens].to(
        next(model.parameters()).device
    )
    if ids.numel() < 2:
        raise ValueError('perplexity requires at least two tokens')

    model.reset_hybrid_routes()
    synchronize()
    start = time.perf_counter()
    total_nll, token_count = model.forward_prefill_loss(
        ids,
        chunk_size=chunk_size,
        max_length=ids.numel(),
    )
    synchronize()
    elapsed = time.perf_counter() - start

    mean_nll = (total_nll / token_count).item()
    return {
        'chunk_size': chunk_size,
        'mean_nll': mean_nll,
        'perplexity': math.exp(mean_nll),
        'seconds': elapsed,
        'tokens_per_second': token_count / elapsed,
        'tokens': token_count,
    }


def run_fast_prefill_quality_sweep(
    dense_model,
    token_ids,
    chunk_sizes=(32, 64, 128, 256),
    max_tokens=2048,
):
    dense_result = evaluate_perplexity(
        dense_model,
        token_ids,
        max_tokens=max_tokens,
    )
    print('--- dense native-context baseline ---')
    print(dense_result)

    routed_config = {
        'block_size': 64,
        'window_size': 256,
        'route_blocks': 16,
        'beam_width': 32,
        'summary_parts': 4,
        'global_blocks': 1,
        'route_refresh_interval': 4,
    }
    results = {'dense': dense_result}

    for routing in ('full_scan', 'hierarchical'):
        clear_gpu_cache()
        routed_model = make_fast_hybrid_sliding_model(
            dense_model,
            routing=routing,
            **routed_config,
        )
        try:
            results[routing] = {}
            for chunk_size in chunk_sizes:
                result = evaluate_fast_hybrid_perplexity(
                    routed_model,
                    token_ids,
                    max_tokens=max_tokens,
                    chunk_size=chunk_size,
                )
                results[routing][chunk_size] = result
                print(
                    '--- fast hybrid quality ---',
                    routing,
                    'chunk_size=',
                    chunk_size,
                    '---',
                )
                print(result)
        finally:
            del routed_model
            clear_gpu_cache()

    return results


RUN_FAST_PREFILL_QUALITY_SWEEP = True
if RUN_FAST_PREFILL_QUALITY_SWEEP:
    missing = [
        name for name in ('model', 'shakespeare_ids')
        if name not in globals()
    ]
    if missing:
        raise RuntimeError(
            'Missing ' + ', '.join(missing) + '. Execute the Pythia '
            'weights and Shakespeare cells first.'
        )

    fast_prefill_quality_results = run_fast_prefill_quality_sweep(
        model,
        shakespeare_ids,
        chunk_sizes=(32, 64, 128, 256),
        max_tokens=2048,
    )

--- dense native-context baseline ---
(3.06980394327064, 21.537679654009136)
--- fast hybrid quality --- full_scan chunk_size= 32 ---
{'chunk_size': 32, 'mean_nll': 3.0746281147003174, 'perplexity': 21.64183213557874, 'seconds': 4.184940729057416, 'tokens_per_second': 489.13476498890594, 'tokens': 2047}
--- fast hybrid quality --- full_scan chunk_size= 64 ---
{'chunk_size': 64, 'mean_nll': 3.0713305473327637, 'perplexity': 21.570584273029787, 'seconds': 2.0757533910218626, 'tokens_per_second': 986.1479734797842, 'tokens': 2047}
--- fast hybrid quality --- full_scan chunk_size= 128 ---
{'chunk_size': 128, 'mean_nll': 3.0715136528015137, 'perplexity': 21.57453432660148, 'seconds': 1.1523143399972469, 'tokens_per_second': 1776.4249987593582, 'tokens': 2047}
--- fast hybrid quality --- full_scan chunk_size= 256 ---
{'chunk_size': 256, 'mean_nll': 3.0702013969421387, 'perplexity': 21.546241585235972, 'seconds': 0.7817397029139102, 'tokens_per_second': 2618.5186608405224, 'tokens': 2047}
---

--- dense native-context baseline ---

(3.06980394327064, 21.537679654009136)

--- fast hybrid quality --- full_scan chunk_size= 32 ---

{'chunk_size': 32, 'mean_nll': 3.0746281147003174, 'perplexity': 21.64183213557874, 'seconds': 4.184940729057416, 'tokens_per_second': 489.13476498890594, 'tokens': 2047}

--- fast hybrid quality --- full_scan chunk_size= 64 ---

{'chunk_size': 64, 'mean_nll': 3.0713305473327637, 'perplexity': 21.570584273029787, 'seconds': 2.0757533910218626, 'tokens_per_second': 986.1479734797842, 'tokens': 2047}

--- fast hybrid quality --- full_scan chunk_size= 128 ---

{'chunk_size': 128, 'mean_nll': 3.0715136528015137, 'perplexity': 21.57453432660148, 'seconds': 1.1523143399972469, 'tokens_per_second': 1776.4249987593582, 'tokens': 2047}

--- fast hybrid quality --- full_scan chunk_size= 256 ---

{'chunk_size': 256, 'mean_nll': 3.0702013969421387, 'perplexity': 21.546241585235972, 'seconds': 0.7817397029139102, 'tokens_per_second': 2618.5186608405224, 'tokens': 2047}

--- fast hybrid quality --- hierarchical chunk_size= 32 ---

{'chunk_size': 32, 'mean_nll': 3.074277877807617, 'perplexity': 21.63425369474149, 'seconds': 7.251350828912109, 'tokens_per_second': 282.29223055080115, 'tokens': 2047}

--- fast hybrid quality --- hierarchical chunk_size= 64 ---

{'chunk_size': 64, 'mean_nll': 3.0722930431365967, 'perplexity': 21.591355864560082, 'seconds': 3.360184635967016, 'tokens_per_second': 609.192714617273, 'tokens': 2047}

--- fast hybrid quality --- hierarchical chunk_size= 128 ---

{'chunk_size': 128, 'mean_nll': 3.0718295574188232, 'perplexity': 21.581350898248097, 'seconds': 1.6129632201045752, 'tokens_per_second': 1269.092794234505, 'tokens': 2047}

--- fast hybrid quality --- hierarchical chunk_size= 256 ---

{'chunk_size': 256, 'mean_nll': 3.0697357654571533, 'perplexity': 21.536211312157484, 'seconds': 0.9976765511091799, 'tokens_per_second': 2051.7671761696924, 'tokens': 2047}

## 22. Very-long-context speed sweep

This is a speed-only experiment. Pythia-1B was trained with a 2048-token
context, so no perplexity or generation-quality claim is made for these
lengths. Dense attention is intentionally not included beyond the existing
14K reference because its prefill memory and compute grow quadratically.

The hybrid models are evaluated separately at increasing prompt lengths. Each
run allocates a fresh preallocated KV-cache, measures chunked prefill and
decode, records peak CUDA memory when available, then releases the cache.

In [31]:
@torch.inference_mode()
def benchmark_fast_long_context_sweep(
    model,
    token_ids,
    prompt_lengths=(14_000, 32_768, 65_536, 131_072),
    new_tokens=64,
    chunk_size=256,
):
    device = next(model.parameters()).device
    results = []

    for prompt_length in prompt_lengths:
        if token_ids.numel() < prompt_length:
            results.append({
                'prompt_length': prompt_length,
                'status': 'not_enough_tokens',
            })
            continue

        clear_gpu_cache()
        if device.type == 'cuda':
            torch.cuda.reset_peak_memory_stats(device)

        try:
            result = benchmark_fast_hybrid_generation(
                model,
                token_ids,
                prompt_length=prompt_length,
                new_tokens=new_tokens,
                chunk_size=chunk_size,
            )
            result['status'] = 'ok'
            result['peak_cuda_allocated_gib'] = (
                torch.cuda.max_memory_allocated(device) / 2**30
                if device.type == 'cuda' else None
            )
            result['peak_cuda_reserved_gib'] = (
                torch.cuda.max_memory_reserved(device) / 2**30
                if device.type == 'cuda' else None
            )
        except RuntimeError as exc:
            message = str(exc)
            if 'out of memory' not in message.lower():
                raise
            result = {
                'prompt_length': prompt_length,
                'status': 'cuda_oom',
                'error': message.split('\\n')[0],
            }
        finally:
            clear_gpu_cache()

        results.append(result)
        print(result)

    return results

### 22.1 Very-long-context full-scan routing

Full-scan routing is measured with the same chunked prefill configuration used
for the hierarchical comparison.

In [34]:
RUN_FAST_FULL_SCAN_LONG_CONTEXT = True

if RUN_FAST_FULL_SCAN_LONG_CONTEXT:
    missing = [
        name for name in ('model', 'shakespeare_ids')
        if name not in globals()
    ]
    if missing:
        raise RuntimeError(
            'Missing ' + ', '.join(missing) + '. Execute the Pythia '
            'weights and Shakespeare cells first.'
        )

    clear_gpu_cache()
    full_scan_long_model = make_fast_hybrid_sliding_model(
        model,
        routing='full_scan',
        **FAST_HYBRID_14K_CONFIG,
    )
    try:
        full_scan_long_results = benchmark_fast_long_context_sweep(
            full_scan_long_model,
            shakespeare_ids,
            prompt_lengths=(14_000, 32_768, 65_536, 131_072),
            new_tokens=3000,
            chunk_size=256,
        )
        print('--- very-long-context full-scan sweep ---')
        print(full_scan_long_results)
    finally:
        del full_scan_long_model
        clear_gpu_cache()

{'prompt_length': 14000, 'chunk_size': 256, 'prefill_seconds': 6.922848785994574, 'prefill_tokens_per_second': 2022.2888629783474, 'decode_seconds': 56.53145436104387, 'decode_tokens_per_second': 53.06780152585843, 'total_seconds': 63.454303147038445, 'status': 'ok', 'peak_cuda_allocated_gib': 9.9633469581604, 'peak_cuda_reserved_gib': 10.828125}
{'prompt_length': 32768, 'chunk_size': 256, 'prefill_seconds': 16.678438908187672, 'prefill_tokens_per_second': 1964.6922700849263, 'decode_seconds': 70.15705623314716, 'decode_tokens_per_second': 42.7612012401197, 'total_seconds': 86.83549514133483, 'status': 'ok', 'peak_cuda_allocated_gib': 12.32784366607666, 'peak_cuda_reserved_gib': 13.236328125}
{'prompt_length': 65536, 'chunk_size': 256, 'prefill_seconds': 34.24089079396799, 'prefill_tokens_per_second': 1913.968897431403, 'decode_seconds': 63.13428820902482, 'decode_tokens_per_second': 47.517760714551954, 'total_seconds': 97.37517900299281, 'status': 'ok', 'peak_cuda_allocated_gib': 16.4

{'prompt_length': 14000, 'chunk_size': 256, 'prefill_seconds': 6.922848785994574, 'prefill_tokens_per_second': 2022.2888629783474, 'decode_seconds': 56.53145436104387, 'decode_tokens_per_second': 53.06780152585843, 'total_seconds': 63.454303147038445, 'status': 'ok', 'peak_cuda_allocated_gib': 9.9633469581604, 'peak_cuda_reserved_gib': 10.828125}

{'prompt_length': 32768, 'chunk_size': 256, 'prefill_seconds': 16.678438908187672, 'prefill_tokens_per_second': 1964.6922700849263, 'decode_seconds': 70.15705623314716, 'decode_tokens_per_second': 42.7612012401197, 'total_seconds': 86.83549514133483, 'status': 'ok', 'peak_cuda_allocated_gib': 12.32784366607666, 'peak_cuda_reserved_gib': 13.236328125}

{'prompt_length': 65536, 'chunk_size': 256, 'prefill_seconds': 34.24089079396799, 'prefill_tokens_per_second': 1913.968897431403, 'decode_seconds': 63.13428820902482, 'decode_tokens_per_second': 47.517760714551954, 'total_seconds': 97.37517900299281, 'status': 'ok', 'peak_cuda_allocated_gib': 16.45736026763916, 'peak_cuda_reserved_gib': 17.3359375}

{'prompt_length': 131072, 'chunk_size': 256, 'prefill_seconds': 69.95123011199757, 'prefill_tokens_per_second': 1873.7626170425185, 'decode_seconds': 72.69605531799607, 'decode_tokens_per_second': 41.26771372775358, 'total_seconds': 142.64728542999364, 'status': 'ok', 'peak_cuda_allocated_gib': 24.723413944244385, 'peak_cuda_reserved_gib': 25.578125}

### 22.2 Very-long-context hierarchical routing

This cell uses exactly the same prompt lengths, chunk size, cache allocation,
and decode length as the full-scan cell.

In [35]:
RUN_FAST_HIERARCHICAL_LONG_CONTEXT = True

if RUN_FAST_HIERARCHICAL_LONG_CONTEXT:
    missing = [
        name for name in ('model', 'shakespeare_ids')
        if name not in globals()
    ]
    if missing:
        raise RuntimeError(
            'Missing ' + ', '.join(missing) + '. Execute the Pythia '
            'weights and Shakespeare cells first.'
        )

    clear_gpu_cache()
    hierarchical_long_model = make_fast_hybrid_sliding_model(
        model,
        routing='hierarchical',
        **FAST_HYBRID_14K_CONFIG,
    )
    try:
        hierarchical_long_results = benchmark_fast_long_context_sweep(
            hierarchical_long_model,
            shakespeare_ids,
            prompt_lengths=(14_000, 32_768, 65_536, 131_072),
            new_tokens=3000,
            chunk_size=256,
        )
        print('--- very-long-context hierarchical sweep ---')
        print(hierarchical_long_results)
    finally:
        del hierarchical_long_model
        clear_gpu_cache()

{'prompt_length': 14000, 'chunk_size': 256, 'prefill_seconds': 10.264678908046335, 'prefill_tokens_per_second': 1363.9004322897622, 'decode_seconds': 66.32514774706215, 'decode_tokens_per_second': 45.231712282659544, 'total_seconds': 76.58982665510848, 'status': 'ok', 'peak_cuda_allocated_gib': 9.9633469581604, 'peak_cuda_reserved_gib': 10.828125}
{'prompt_length': 32768, 'chunk_size': 256, 'prefill_seconds': 22.602459785994142, 'prefill_tokens_per_second': 1449.7537131027236, 'decode_seconds': 59.84187023178674, 'decode_tokens_per_second': 50.13212301654408, 'total_seconds': 82.44433001778089, 'status': 'ok', 'peak_cuda_allocated_gib': 12.32784366607666, 'peak_cuda_reserved_gib': 13.236328125}
{'prompt_length': 65536, 'chunk_size': 256, 'prefill_seconds': 47.270340671995655, 'prefill_tokens_per_second': 1386.40845545726, 'decode_seconds': 61.17590872081928, 'decode_tokens_per_second': 49.038911930033095, 'total_seconds': 108.44624939281493, 'status': 'ok', 'peak_cuda_allocated_gib': 1

{'prompt_length': 14000, 'chunk_size': 256, 'prefill_seconds': 10.264678908046335, 'prefill_tokens_per_second': 1363.9004322897622, 'decode_seconds': 66.32514774706215, 'decode_tokens_per_second': 45.231712282659544, 'total_seconds': 76.58982665510848, 'status': 'ok', 'peak_cuda_allocated_gib': 9.9633469581604, 'peak_cuda_reserved_gib': 10.828125}

{'prompt_length': 32768, 'chunk_size': 256, 'prefill_seconds': 22.602459785994142, 'prefill_tokens_per_second': 1449.7537131027236, 'decode_seconds': 59.84187023178674, 'decode_tokens_per_second': 50.13212301654408, 'total_seconds': 82.44433001778089, 'status': 'ok', 'peak_cuda_allocated_gib': 12.32784366607666, 'peak_cuda_reserved_gib': 13.236328125}

{'prompt_length': 65536, 'chunk_size': 256, 'prefill_seconds': 47.270340671995655, 'prefill_tokens_per_second': 1386.40845545726, 'decode_seconds': 61.17590872081928, 'decode_tokens_per_second': 49.038911930033095, 'total_seconds': 108.44624939281493, 'status': 'ok', 'peak_cuda_allocated_gib': 16.45736026763916, 'peak_cuda_reserved_gib': 17.3359375}

{'prompt_length': 131072, 'chunk_size': 256, 'prefill_seconds': 97.97639987594448, 'prefill_tokens_per_second': 1337.7915514956705, 'decode_seconds': 62.18208728102036, 'decode_tokens_per_second': 48.245405247367444, 'total_seconds': 160.15848715696484, 'status': 'ok', 'peak_cuda_allocated_gib': 24.723413944244385, 'peak_cuda_reserved_gib': 25.578125}

На контекстах до 128K full-scan пока быстрее hierarchical, а prefill обеих реализаций масштабируется практически линейно. Иерархическое преимущество существует теоретически, но текущая реализация его ещё не реализует на уровне реального времени.

## 19. Bounded KV-cache для million-token speed test

Полный KV-cache растёт как `O(N)`. В экспериментальном bounded-режиме точные K/V сохраняются только для локального окна, а завершённые блоки истории сжимаются в иерархические representative K/V. Число сегментов истории растёт как `O(log N)`, поэтому cache не занимает память пропорционально всему контексту.

Это speed-only прототип: усреднение K/V может ухудшить дальний поиск, поэтому PPL будет проверяться отдельно.

In [38]:
class BoundedKVCache:
    def __init__(self, config, max_length, block_size=256, local_window=256, memory_slots=16, device=None, dtype=None):
        self.max_length = int(max_length)
        self.block_size = int(block_size)
        self.local_window = int(local_window)
        self.memory_slots = int(memory_slots)
        self.device = device or DEVICE
        self.dtype = dtype or DTYPE
        self.num_heads = config.num_attention_heads
        self.head_dim = config.head_dim
        self.local_key = torch.empty(1, self.num_heads, self.local_window, self.head_dim, device=self.device, dtype=self.dtype)
        self.local_value = torch.empty_like(self.local_key)
        max_blocks = max(1, math.ceil(self.max_length / self.block_size))
        self.max_levels = max(2, int(math.log2(max_blocks)) + 2)
        self.segment_key = torch.zeros(1, self.num_heads, self.max_levels, self.head_dim, device=self.device, dtype=self.dtype)
        self.segment_value = torch.zeros_like(self.segment_key)
        self.segment_weight = torch.zeros(self.max_levels, device=self.device, dtype=torch.float32)
        self.active_levels = [False] * self.max_levels
        self.active_indices = torch.empty(0, device=self.device, dtype=torch.long)
        self.segment_weights = [0.0] * self.max_levels
        self.current_key_sum = torch.zeros(self.num_heads, self.head_dim, device=self.device, dtype=torch.float32)
        self.current_value_sum = torch.zeros_like(self.current_key_sum)
        self.current_count = 0

    @torch.no_grad()
    def _commit_current_block(self):
        if self.current_count == 0:
            return
        carry_key = self.current_key_sum / float(self.current_count)
        carry_value = self.current_value_sum / float(self.current_count)
        carry_weight = float(self.current_count)
        self.current_key_sum.zero_()
        self.current_value_sum.zero_()
        self.current_count = 0
        for level in range(self.max_levels):
            if not self.active_levels[level]:
                self.segment_key[0, :, level, :].copy_(carry_key.to(self.dtype))
                self.segment_value[0, :, level, :].copy_(carry_value.to(self.dtype))
                self.segment_weight[level] = carry_weight
                self.segment_weights[level] = carry_weight
                self.active_levels[level] = True
                self.active_indices = torch.tensor([i for i, active in enumerate(self.active_levels) if active], device=self.device, dtype=torch.long)
                return
            old_weight = self.segment_weights[level]
            total_weight = old_weight + carry_weight
            old_key = self.segment_key[0, :, level, :].float()
            old_value = self.segment_value[0, :, level, :].float()
            carry_key = (old_key * old_weight + carry_key * carry_weight) / total_weight
            carry_value = (old_value * old_weight + carry_value * carry_weight) / total_weight
            carry_weight = total_weight
            self.segment_weights[level] = 0.0
            self.active_levels[level] = False
        raise RuntimeError('Недостаточно уровней bounded KV-cache')

    @torch.no_grad()
    def append(self, key, value, position):
        if position >= self.max_length:
            raise IndexError('position превышает max_length bounded KV-cache')
        if position > 0 and position % self.block_size == 0:
            self._commit_current_block()
        slot = position % self.local_window
        self.local_key[:, :, slot:slot + 1, :].copy_(key)
        self.local_value[:, :, slot:slot + 1, :].copy_(value)
        self.current_key_sum.add_(key[0, :, 0, :].float())
        self.current_value_sum.add_(value[0, :, 0, :].float())
        self.current_count += 1

    def attention_kv(self, query, position):
        local_start = max(0, position - self.local_window + 1)
        local_positions = torch.arange(local_start, position + 1, device=self.device, dtype=torch.long)
        local_slots = local_positions.remainder(self.local_window)
        local_key = self.local_key.index_select(2, local_slots)
        local_value = self.local_value.index_select(2, local_slots)
        active = self.active_indices
        if active.numel() == 0 or self.memory_slots == 0:
            return local_key, local_value
        memory_key = self.segment_key.index_select(2, active)
        memory_value = self.segment_value.index_select(2, active)
        scores = torch.einsum('bhqd,bhkd->bhqk', query.float(), memory_key.float())[:, :, 0, :]
        keep = min(self.memory_slots, memory_key.shape[2])
        selected = torch.topk(scores, k=keep, dim=-1, largest=True, sorted=False).indices
        gather_ids = selected.unsqueeze(-1).expand(-1, -1, -1, self.head_dim)
        selected_key = memory_key.gather(2, gather_ids)
        selected_value = memory_value.gather(2, gather_ids)
        return torch.cat((local_key, selected_key), dim=2), torch.cat((local_value, selected_value), dim=2)


class BoundedOceanAttention(PythiaAttention):
    def __init__(self, config, block_size=256, local_window=256, memory_slots=16):
        super().__init__(config)
        self.block_size = block_size
        self.local_window = local_window
        self.memory_slots = memory_slots

    @torch.inference_mode()
    def forward_token(self, hidden_states, cache, position):
        if hidden_states.shape[0] != 1 or hidden_states.shape[1] != 1:
            raise NotImplementedError('bounded path поддерживает только batch=1, q_len=1')
        qkv = self.query_key_value(hidden_states)
        qkv = qkv.view(1, 1, self.num_attention_heads, 3 * self.head_dim).transpose(1, 2)
        query, key, value = qkv.chunk(3, dim=-1)
        position_ids = torch.tensor([position], device=hidden_states.device, dtype=torch.long)
        cos, sin = self.rotary_emb(position_ids, hidden_states.dtype)
        query, key = apply_rotary(query, key, cos, sin, self.rotary_ndims)
        cache.append(key, value, position)
        route_key, route_value = cache.attention_kv(query, position)
        attention_output = F.scaled_dot_product_attention(query, route_key, route_value, dropout_p=0.0, is_causal=False)
        attention_output = attention_output.transpose(1, 2).contiguous().view(1, 1, -1)
        return self.dense(attention_output)


class BoundedOceanPythiaForCausalLM(PythiaForCausalLM):
    def __init__(self, config, block_size=256, local_window=256, memory_slots=16):
        super().__init__(config)
        self.bounded_kwargs = {'block_size': block_size, 'local_window': local_window, 'memory_slots': memory_slots}
        for layer in self.gpt_neox.layers:
            layer.attention = BoundedOceanAttention(config, **self.bounded_kwargs)

    def new_bounded_cache(self, max_length):
        return [BoundedKVCache(self.config, max_length=max_length, device=DEVICE, dtype=DTYPE, **self.bounded_kwargs) for _ in self.gpt_neox.layers]

    @torch.inference_mode()
    def forward_bounded_token(self, input_ids, caches, position):
        hidden_states = self.gpt_neox.embed_in(input_ids.view(1, 1))
        for layer, cache in zip(self.gpt_neox.layers, caches):
            residual = hidden_states
            attention_input = layer.input_layernorm(hidden_states)
            attention_output = layer.attention.forward_token(attention_input, cache, position)
            if layer.use_parallel_residual:
                mlp_input = layer.post_attention_layernorm(hidden_states)
                hidden_states = residual + attention_output + layer.mlp(mlp_input)
            else:
                hidden_states = residual + attention_output
                hidden_states = hidden_states + layer.mlp(layer.post_attention_layernorm(hidden_states))
        hidden_states = self.gpt_neox.final_layer_norm(hidden_states)
        return self.embed_out(hidden_states)


def make_bounded_ocean_model(dense_model, block_size=256, local_window=256, memory_slots=16):
    bounded = BoundedOceanPythiaForCausalLM(dense_model.config, block_size=block_size, local_window=local_window, memory_slots=memory_slots)
    bounded.load_state_dict(dense_model.state_dict(), strict=True)
    return bounded.to(device=DEVICE, dtype=DTYPE).eval()


def repeated_prompt(token_ids, length):
    source = token_ids.flatten().to(DEVICE)
    repeats = math.ceil(length / source.numel())
    return source.repeat(repeats)[:length]


@torch.inference_mode()
def benchmark_bounded_generation(model, token_ids, prompt_length=1_000_000, new_tokens=16):
    ids = repeated_prompt(token_ids, prompt_length)
    caches = model.new_bounded_cache(prompt_length + new_tokens)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    synchronize()
    start = time.perf_counter()
    logits = None
    for position, token in enumerate(ids):
        logits = model.forward_bounded_token(token, caches, position)
        if (position + 1) % 10_000 == 0:
            print(f'prefill progress: {position + 1:,}/{prompt_length:,}')
    synchronize()
    prefill_seconds = time.perf_counter() - start
    next_token = logits[:, -1:, :].argmax(dim=-1)
    synchronize()
    decode_start = time.perf_counter()
    for position in range(prompt_length, prompt_length + new_tokens):
        logits = model.forward_bounded_token(next_token[:, 0], caches, position)
        next_token = logits[:, -1:, :].argmax(dim=-1)
    synchronize()
    decode_seconds = time.perf_counter() - decode_start
    result = {'prompt_length': prompt_length, 'new_tokens': new_tokens, 'prefill_seconds': prefill_seconds, 'prefill_tokens_per_second': prompt_length / prefill_seconds, 'decode_seconds': decode_seconds, 'decode_tokens_per_second': new_tokens / decode_seconds, 'total_seconds': prefill_seconds + decode_seconds, 'cache': 'bounded_local_plus_logarithmic_segments', 'block_size': model.bounded_kwargs['block_size'], 'local_window': model.bounded_kwargs['local_window'], 'memory_slots': model.bounded_kwargs['memory_slots']}
    if torch.cuda.is_available():
        result['peak_cuda_allocated_gib'] = torch.cuda.max_memory_allocated() / 2**30
        result['peak_cuda_reserved_gib'] = torch.cuda.max_memory_reserved() / 2**30
    return result

In [39]:
# Speed-only тест; PPL на bounded cache будет добавлен отдельно.
RUN_BOUNDED_1M_BENCHMARK = True
BOUNDED_CONFIG = {'block_size': 256, 'local_window': 256, 'memory_slots': 16}

if RUN_BOUNDED_1M_BENCHMARK:
    if 'model' not in globals() or 'shakespeare_ids' not in globals():
        raise RuntimeError('Сначала загрузите model и shakespeare_ids')
    clear_gpu_cache()
    bounded_model = make_bounded_ocean_model(model, **BOUNDED_CONFIG)
    try:
        bounded_1m_result = benchmark_bounded_generation(bounded_model, shakespeare_ids, prompt_length=1_000_000, new_tokens=16)
        print('--- bounded KV 1M speed ---')
        print(bounded_1m_result)
    finally:
        del bounded_model
        clear_gpu_cache()

KeyboardInterrupt: 

prefill progress: 100,000/1,000,000 - на 33 минуте

## 20. Chunked prefill для bounded KV-cache

Token-by-token prefill оставлен только для decode и отладки. В этом режиме prompt делится на chunks, а causal attention строится внутри chunk одним вызовом SDPA. Это сохраняет ограниченный cache и убирает миллион Python-вызовов.

In [40]:
@torch.no_grad()
def _bounded_append_chunk(self, key, value, start_position):
    q_len = key.shape[2]
    offset = 0
    while offset < q_len:
        position = start_position + offset
        if position > 0 and position % self.block_size == 0:
            self._commit_current_block()
        take = min(q_len - offset, self.block_size - (position % self.block_size))
        positions = torch.arange(position, position + take, device=self.device, dtype=torch.long)
        slots = positions.remainder(self.local_window)
        self.local_key.index_copy_(2, slots, key[:, :, offset:offset + take, :])
        self.local_value.index_copy_(2, slots, value[:, :, offset:offset + take, :])
        self.current_key_sum.add_(key[0, :, offset:offset + take, :].float().sum(dim=1))
        self.current_value_sum.add_(value[0, :, offset:offset + take, :].float().sum(dim=1))
        self.current_count += take
        offset += take


BoundedKVCache.append_chunk = _bounded_append_chunk


@torch.inference_mode()
def _bounded_forward_chunk(self, hidden_states, cache, start_position):
    if hidden_states.shape[0] != 1:
        raise NotImplementedError('chunked bounded path поддерживает только batch=1')
    q_len = hidden_states.shape[1]
    qkv = self.query_key_value(hidden_states)
    qkv = qkv.view(1, q_len, self.num_attention_heads, 3 * self.head_dim).transpose(1, 2)
    query, key, value = qkv.chunk(3, dim=-1)
    position_ids = torch.arange(start_position, start_position + q_len, device=hidden_states.device, dtype=torch.long)
    cos, sin = self.rotary_emb(position_ids, hidden_states.dtype)
    query, key = apply_rotary(query, key, cos, sin, self.rotary_ndims)
    past_key, past_value = cache.attention_kv(query[:, :, -1:, :], start_position - 1)
    past_len = past_key.shape[2]
    route_key = torch.cat((past_key, key), dim=2)
    route_value = torch.cat((past_value, value), dim=2)
    allowed = torch.ones(q_len, past_len + q_len, device=hidden_states.device, dtype=torch.bool)
    allowed[:, past_len:] = torch.tril(torch.ones(q_len, q_len, device=hidden_states.device, dtype=torch.bool))
    attention_output = F.scaled_dot_product_attention(query, route_key, route_value, attn_mask=allowed[None, None, :, :], dropout_p=0.0, is_causal=False)
    cache.append_chunk(key, value, start_position)
    attention_output = attention_output.transpose(1, 2).contiguous().view(1, q_len, -1)
    return self.dense(attention_output)


BoundedOceanAttention.forward_chunk = _bounded_forward_chunk


@torch.inference_mode()
def _bounded_forward_bounded_chunk(self, input_ids, caches, start_position):
    hidden_states = self.gpt_neox.embed_in(input_ids.view(1, -1))
    for layer, cache in zip(self.gpt_neox.layers, caches):
        residual = hidden_states
        attention_input = layer.input_layernorm(hidden_states)
        attention_output = layer.attention.forward_chunk(attention_input, cache, start_position)
        if layer.use_parallel_residual:
            mlp_input = layer.post_attention_layernorm(hidden_states)
            hidden_states = residual + attention_output + layer.mlp(mlp_input)
        else:
            hidden_states = residual + attention_output
            hidden_states = hidden_states + layer.mlp(layer.post_attention_layernorm(hidden_states))
    hidden_states = self.gpt_neox.final_layer_norm(hidden_states)
    return self.embed_out(hidden_states)


BoundedOceanPythiaForCausalLM.forward_bounded_chunk = _bounded_forward_bounded_chunk


@torch.inference_mode()
def benchmark_bounded_chunked_generation(model, token_ids, prompt_length=1_000_000, new_tokens=16, chunk_size=256):
    ids = repeated_prompt(token_ids, prompt_length)
    caches = model.new_bounded_cache(prompt_length + new_tokens)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    synchronize()
    start = time.perf_counter()
    logits = None
    for chunk_start in range(0, prompt_length, chunk_size):
        chunk = ids[chunk_start:min(chunk_start + chunk_size, prompt_length)]
        logits = model.forward_bounded_chunk(chunk, caches, chunk_start)
        if (chunk_start + chunk.numel()) % 100_000 < chunk_size:
            print(f'chunked prefill progress: {chunk_start + chunk.numel():,}/{prompt_length:,}')
    synchronize()
    prefill_seconds = time.perf_counter() - start
    next_token = logits[:, -1:, :].argmax(dim=-1)
    synchronize()
    decode_start = time.perf_counter()
    for position in range(prompt_length, prompt_length + new_tokens):
        logits = model.forward_bounded_token(next_token[:, 0], caches, position)
        next_token = logits[:, -1:, :].argmax(dim=-1)
    synchronize()
    decode_seconds = time.perf_counter() - decode_start
    result = {'prompt_length': prompt_length, 'chunk_size': chunk_size, 'new_tokens': new_tokens, 'prefill_seconds': prefill_seconds, 'prefill_tokens_per_second': prompt_length / prefill_seconds, 'decode_seconds': decode_seconds, 'decode_tokens_per_second': new_tokens / decode_seconds, 'total_seconds': prefill_seconds + decode_seconds, 'cache': 'bounded_local_plus_logarithmic_segments'}
    if torch.cuda.is_available():
        result['peak_cuda_allocated_gib'] = torch.cuda.max_memory_allocated() / 2**30
        result['peak_cuda_reserved_gib'] = torch.cuda.max_memory_reserved() / 2**30
    return result

In [41]:
# Chunked speed-only benchmark. Начать лучше с 32K или 128K, затем 1M.
RUN_BOUNDED_CHUNKED_1M_BENCHMARK = True
BOUNDED_CHUNKED_PROMPT_LENGTH = 32_768  # сначала проверка; затем 1_000_000
CHUNKED_BOUNDED_CONFIG = {'block_size': 256, 'local_window': 256, 'memory_slots': 16, 'chunk_size': 256}

if RUN_BOUNDED_CHUNKED_1M_BENCHMARK:
    if 'model' not in globals() or 'shakespeare_ids' not in globals():
        raise RuntimeError('Сначала загрузите model и shakespeare_ids')
    clear_gpu_cache()
    chunked_bounded_model = make_bounded_ocean_model(model, block_size=CHUNKED_BOUNDED_CONFIG['block_size'], local_window=CHUNKED_BOUNDED_CONFIG['local_window'], memory_slots=CHUNKED_BOUNDED_CONFIG['memory_slots'])
    try:
        chunked_1m_result = benchmark_bounded_chunked_generation(chunked_bounded_model, shakespeare_ids, prompt_length=BOUNDED_CHUNKED_PROMPT_LENGTH, new_tokens=16, chunk_size=CHUNKED_BOUNDED_CONFIG['chunk_size'])
        print('--- chunked bounded KV 1M speed ---')
        print(chunked_1m_result)
    finally:
        del chunked_bounded_model
        clear_gpu_cache()

--- chunked bounded KV 1M speed ---
{'prompt_length': 32768, 'chunk_size': 256, 'new_tokens': 16, 'prefill_seconds': 8.840559411095455, 'prefill_tokens_per_second': 3706.5527729924092, 'decode_seconds': 0.3066619529854506, 'decode_tokens_per_second': 52.17471500535024, 'total_seconds': 9.147221364080906, 'cache': 'bounded_local_plus_logarithmic_segments', 'peak_cuda_allocated_gib': 7.702514171600342, 'peak_cuda_reserved_gib': 7.81640625}


--- chunked bounded KV 1M speed ---

{'prompt_length': 32768, 'chunk_size': 256, 'new_tokens': 16, 'prefill_seconds': 8.840559411095455, 'prefill_tokens_per_second': 3706.5527729924092, 'decode_seconds': 0.3066619529854506, 'decode_tokens_per_second': 52.17471500535024, 'total_seconds': 9.147221364080906, 'cache': 'bounded_local_plus_logarithmic_segments', 'peak_cuda_allocated_gib': 7.702514171600342, 'peak_cuda_reserved_gib': 7.81640625}

In [42]:
# Chunked speed-only benchmark. Начать лучше с 32K или 128K, затем 1M.
RUN_BOUNDED_CHUNKED_1M_BENCHMARK = True
BOUNDED_CHUNKED_PROMPT_LENGTH = 1_000_000  # сначала проверка; затем 1_000_000
CHUNKED_BOUNDED_CONFIG = {'block_size': 256, 'local_window': 256, 'memory_slots': 16, 'chunk_size': 256}

if RUN_BOUNDED_CHUNKED_1M_BENCHMARK:
    if 'model' not in globals() or 'shakespeare_ids' not in globals():
        raise RuntimeError('Сначала загрузите model и shakespeare_ids')
    clear_gpu_cache()
    chunked_bounded_model = make_bounded_ocean_model(model, block_size=CHUNKED_BOUNDED_CONFIG['block_size'], local_window=CHUNKED_BOUNDED_CONFIG['local_window'], memory_slots=CHUNKED_BOUNDED_CONFIG['memory_slots'])
    try:
        chunked_1m_result = benchmark_bounded_chunked_generation(chunked_bounded_model, shakespeare_ids, prompt_length=BOUNDED_CHUNKED_PROMPT_LENGTH, new_tokens=16, chunk_size=CHUNKED_BOUNDED_CONFIG['chunk_size'])
        print('--- chunked bounded KV 1M speed ---')
        print(chunked_1m_result)
    finally:
        del chunked_bounded_model
        clear_gpu_cache()

chunked prefill progress: 100,096/1,000,000
chunked prefill progress: 200,192/1,000,000
chunked prefill progress: 300,032/1,000,000
chunked prefill progress: 400,128/1,000,000
chunked prefill progress: 500,224/1,000,000
chunked prefill progress: 600,064/1,000,000
chunked prefill progress: 700,160/1,000,000
chunked prefill progress: 800,000/1,000,000
chunked prefill progress: 900,096/1,000,000
chunked prefill progress: 1,000,000/1,000,000
--- chunked bounded KV 1M speed ---
{'prompt_length': 1000000, 'chunk_size': 256, 'new_tokens': 16, 'prefill_seconds': 274.34823948983103, 'prefill_tokens_per_second': 3645.0024314337397, 'decode_seconds': 0.2961924457922578, 'decode_tokens_per_second': 54.01893339042824, 'total_seconds': 274.6444319356233, 'cache': 'bounded_local_plus_logarithmic_segments', 'peak_cuda_allocated_gib': 7.708284378051758, 'peak_cuda_reserved_gib': 7.81640625}


--- chunked bounded KV 1M speed ---

{'prompt_length': 1000000, 'chunk_size': 256, 'new_tokens': 16, 'prefill_seconds': 274.34823948983103, 'prefill_tokens_per_second': 3645.0024314337397, 'decode_seconds': 0.2961924457922578, 'decode_tokens_per_second': 54.01893339042824, 'total_seconds': 274.6444319356233, 'cache': 'bounded_local_plus_logarithmic_segments', 'peak_cuda_allocated_gib': 7.708284378051758, 'peak_cuda_reserved_gib': 7.81640625}

Это подтверждает, что текущий bounded-cache действительно удерживает память практически постоянной и даёт примерно:
- память: O(W + log N);
- prefill compute: практически O(N) при фиксированном размере chunk;
- decode attention: O(W + log N) или почти O(1) при ограниченном числе summary-сегментов.
Но это пока только speed benchmark. Он не доказывает, что модель сохраняет качество на миллионе токенов: prompt повторяет Shakespeare, а дальняя история заменяется усреднёнными representative K/V. Следующий обязательный эксперимент — тест дальнего поиска или хотя бы controlled retrieval на 32K/128K/1M.

## 21. Controlled long-context retrieval: needle-in-a-haystack

В prompt вставляется уникальный дальний код, затем в конце задаётся вопрос о нём. Метрики: NLL правильного ответа, perplexity ответа, exact-match и фактически сгенерированный ответ.

In [43]:
def make_needle_prompt(tokenizer, filler_ids, prompt_length, needle_fraction=0.25):
    prefix = tokenizer('Archive record begins.\n', add_special_tokens=False, return_tensors='pt').input_ids[0].to(DEVICE)
    needle_text = 'The unique archival retrieval code is ORBIT-314159.'
    needle = tokenizer(needle_text, add_special_tokens=False, return_tensors='pt').input_ids[0].to(DEVICE)
    answer = tokenizer('ORBIT-314159', add_special_tokens=False, return_tensors='pt').input_ids[0].to(DEVICE)
    suffix = tokenizer('\n\nQuestion: What is the unique archival retrieval code?\nAnswer:', add_special_tokens=False, return_tensors='pt').input_ids[0].to(DEVICE)
    filler_count = prompt_length - prefix.numel() - needle.numel() - suffix.numel()
    if filler_count <= 0:
        raise ValueError('prompt_length слишком мал для needle prompt')
    filler = repeated_prompt(filler_ids, filler_count)
    before = int(filler_count * needle_fraction)
    prompt = torch.cat((prefix, filler[:before], needle, filler[before:], suffix), dim=0)
    return prompt, answer, prefix.numel() + before


@torch.inference_mode()
def evaluate_bounded_needle(model, tokenizer, filler_ids, prompt_length=32_768, chunk_size=256):
    prompt, answer_ids, needle_position = make_needle_prompt(tokenizer, filler_ids, prompt_length)
    caches = model.new_bounded_cache(prompt.numel() + answer_ids.numel())
    synchronize()
    prefill_start = time.perf_counter()
    logits = None
    for chunk_start in range(0, prompt.numel(), chunk_size):
        chunk = prompt[chunk_start:min(chunk_start + chunk_size, prompt.numel())]
        logits = model.forward_bounded_chunk(chunk, caches, chunk_start)
    synchronize()
    prefill_seconds = time.perf_counter() - prefill_start
    total_nll = torch.zeros((), device=DEVICE, dtype=torch.float32)
    predicted_ids = []
    synchronize()
    probe_start = time.perf_counter()
    for offset, target in enumerate(answer_ids):
        prediction = logits[:, -1, :].float().argmax(dim=-1)[0]
        predicted_ids.append(prediction)
        total_nll.add_(-F.log_softmax(logits[:, -1, :].float(), dim=-1)[0, target])
        logits = model.forward_bounded_token(target, caches, prompt.numel() + offset)
    synchronize()
    probe_seconds = time.perf_counter() - probe_start
    predicted = torch.stack(predicted_ids)
    mean_nll = (total_nll / answer_ids.numel()).item()
    result = {'prompt_length': prompt_length, 'needle_position': needle_position, 'answer_tokens': int(answer_ids.numel()), 'answer_mean_nll': mean_nll, 'answer_perplexity': math.exp(mean_nll), 'exact_match': bool(torch.equal(predicted, answer_ids)), 'predicted_answer': tokenizer.decode(predicted.cpu().tolist()), 'target_answer': tokenizer.decode(answer_ids.cpu().tolist()), 'prefill_seconds': prefill_seconds, 'prefill_tokens_per_second': prompt.numel() / prefill_seconds, 'probe_seconds': probe_seconds}
    del caches
    return result


def run_bounded_needle_sweep(model, tokenizer, filler_ids, prompt_lengths=(32_768, 131_072, 1_000_000), chunk_size=256):
    results = []
    for length in prompt_lengths:
        clear_gpu_cache()
        result = evaluate_bounded_needle(model, tokenizer, filler_ids, prompt_length=length, chunk_size=chunk_size)
        results.append(result)
        print(result)
    return results

In [45]:
RUN_BOUNDED_NEEDLE_SWEEP = True

if RUN_BOUNDED_NEEDLE_SWEEP:
    if 'model' not in globals() or 'tokenizer' not in globals() or 'shakespeare_ids' not in globals():
        raise RuntimeError('Сначала загрузите model, tokenizer и shakespeare_ids')
    clear_gpu_cache()
    needle_model = make_bounded_ocean_model(model, block_size=256, local_window=256, memory_slots=16)
    try:
        bounded_needle_results = run_bounded_needle_sweep(needle_model, tokenizer, shakespeare_ids, prompt_lengths=(100, 1_000), chunk_size=256)
        print('--- bounded KV needle retrieval sweep ---')
        print(bounded_needle_results)
    finally:
        del needle_model
        clear_gpu_cache()

{'prompt_length': 100, 'needle_position': 21, 'answer_tokens': 5, 'answer_mean_nll': 1.1248581409454346, 'answer_perplexity': 3.0797799232596503, 'exact_match': False, 'predicted_answer': ' ORBIT-314159', 'target_answer': 'ORBIT-314159', 'prefill_seconds': 0.05626094783656299, 'prefill_tokens_per_second': 1777.431839408361, 'probe_seconds': 0.07715714792720973}
{'prompt_length': 1000, 'needle_position': 246, 'answer_tokens': 5, 'answer_mean_nll': 9.552535057067871, 'answer_perplexity': 14080.343941370968, 'exact_match': False, 'predicted_answer': ' TheOROROR-', 'target_answer': 'ORBIT-314159', 'prefill_seconds': 0.32284393697045743, 'prefill_tokens_per_second': 3097.4718292185466, 'probe_seconds': 0.09390334296040237}
--- bounded KV needle retrieval sweep ---
[{'prompt_length': 100, 'needle_position': 21, 'answer_tokens': 5, 'answer_mean_nll': 1.1248581409454346, 'answer_perplexity': 3.0797799232596503, 'exact_match': False, 'predicted_answer': ' ORBIT-314159', 'target_answer': 'ORBIT-

{'prompt_length': 100, 'needle_position': 21, 'answer_tokens': 5, 'answer_mean_nll': 1.1248581409454346, 'answer_perplexity': 3.0797799232596503, 'exact_match': False, 'predicted_answer': ' ORBIT-314159', 'target_answer': 'ORBIT-314159', 'prefill_seconds': 0.05626094783656299, 'prefill_tokens_per_second': 1777.431839408361, 'probe_seconds': 0.07715714792720973}

{'prompt_length': 1000, 'needle_position': 246, 'answer_tokens': 5, 'answer_mean_nll': 9.552535057067871, 'answer_perplexity': 14080.343941370968, 'exact_match': False, 'predicted_answer': ' TheOROROR-', 'target_answer': 'ORBIT-314159', 'prefill_seconds': 0.32284393697045743, 'prefill_tokens_per_second': 3097.4718292185466, 'probe_seconds': 0.09390334296040237}
